## Gerar Exceis

In [ ]:
# ============================================================
# DEGRADAÇÃO TEMPORAL REALISTA DE CORROSÃO
# - Só cresce anomalias com coluna C == "corrosion"
# - "damage" e "water" mantêm-se inalteradas
# - Gera 20 ficheiros Excel (de 6 em 6 meses, durante 10 anos)
# - Crescimento desacelerado ao longo do tempo
# ============================================================

import re
import math
from pathlib import Path

import numpy as np
import cv2
from openpyxl import load_workbook
from PIL import Image

# ============================================================
# CONFIG
# ============================================================

# Pasta onde está o excel original
PASTA_BASE = r"base_excel"
FICHEIRO_EXCEL = "metricas_anomalias_faces_malha.xlsx"

# Pasta das imagens base para obter largura/altura corretas
PASTA_IMAGENS = "image"

# Pasta onde serão guardados os 20 Excels
PASTA_SAIDA_EXCELS = 'excel_result/degradacao_temporal_excels'

# ------------------------------------------------------------
# PARÂMETROS TEMPORAIS
# ------------------------------------------------------------
NUM_PASSOS = 20              # 20 excels
MESES_POR_PASSO = 6          # de 6 em 6 meses

# ------------------------------------------------------------
# PARÂMETROS DE DEGRADAÇÃO
# ------------------------------------------------------------
# Kernel da dilatação
KERNEL_SIZE = 3

# "Força" inicial de degradação por passo.
# Não precisa de ser inteira. O acumulador trata das frações.
BASE_STEP_STRENGTH = 1.8

# Quanto maior, mais rapidamente a degradação abranda.
# Valores típicos: 1.5 a 4.0
DECAY_RATE = 2.8

# Mínimo de crescimento residual relativo no último passo
# evita que a degradação "morra completamente"
# Ex.: 0.15 = no fim ainda há 15% da força inicial
MIN_RELATIVE_GROWTH = 0.15

# Se True, aplica fechamento morfológico leve após dilatação
# para tapar pequenas falhas/aberturas e deixar o dano mais coeso
APPLY_MORPH_CLOSE = True
MORPH_CLOSE_KERNEL_SIZE = 3

AREA_DECIMALS = 6
EXTENSOES_VALIDAS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def parse_polygon_string(poly_str):
    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()
    if not poly_str:
        return []

    matches = re.findall(r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]", poly_str)
    return [(float(x), float(y)) for x, y in matches]


def polygon_to_string(points):
    return "//".join(f"[{int(round(x))},{int(round(y))}]" for x, y in points)


def polygon_area_pixels(points):
    if len(points) < 3:
        return 0.0

    pts = np.array(points, dtype=float)
    x = pts[:, 0]
    y = pts[:, 1]
    area = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
    return float(area)


def safe_float(v, default=0.0):
    try:
        return float(v)
    except:
        return default


def normalize_anomaly_type(v):
    if v is None:
        return ""
    return str(v).strip().lower()


def index_images_by_name(root_folder):
    root = Path(root_folder)
    image_map = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[p.name.lower()] = p
    return image_map


def get_image_size_cached(image_name, image_index, size_cache):
    if image_name in size_cache:
        return size_cache[image_name]

    img_path = image_index.get(image_name)
    if img_path is None:
        return None

    with Image.open(img_path) as img_ref:
        size_cache[image_name] = img_ref.size  # (width, height)

    return size_cache[image_name]


def polygon_to_mask(points, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)

    pts = np.array(
        [[
            [
                min(max(int(round(x)), 0), width - 1),
                min(max(int(round(y)), 0), height - 1)
            ]
            for x, y in points
        ]],
        dtype=np.int32
    )

    if pts.shape[1] >= 3:
        cv2.fillPoly(mask, pts, 255)

    return mask


def dilate_mask(mask, kernel_size=3, iterations=1):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=iterations)
    return dilated


def close_mask(mask, kernel_size=3):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)


def mask_to_polygon(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return []

    contour = max(contours, key=cv2.contourArea)

    epsilon = 0.002 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)

    points = [(float(pt[0][0]), float(pt[0][1])) for pt in approx]
    return points


def get_step_strength(step_idx, total_steps, base_strength, decay_rate, min_relative_growth):
    """
    step_idx: 1..total_steps
    Retorna força relativa decrescente ao longo do tempo.
    No início ~1.0
    No fim ~min_relative_growth
    """
    if total_steps <= 1:
        return base_strength

    s = (step_idx - 1) / (total_steps - 1)  # 0..1

    # exponencial decrescente normalizada
    relative = math.exp(-decay_rate * s)

    # remap para nunca cair abaixo de min_relative_growth
    relative = min_relative_growth + (1.0 - min_relative_growth) * relative

    return base_strength * relative


def build_output_filename(base_name, step_idx, months_elapsed):
    stem = Path(base_name).stem
    suffix = Path(base_name).suffix
    years = months_elapsed // 12
    rem_months = months_elapsed % 12
    return f"{stem}_degradacao_t{step_idx:02d}_{years:02d}a_{rem_months:02d}m{suffix}"


# ============================================================
# PREPARAÇÃO
# ============================================================

excel_path = Path(PASTA_BASE) / FICHEIRO_EXCEL
output_dir = Path(PASTA_SAIDA_EXCELS)
output_dir.mkdir(parents=True, exist_ok=True)

if not excel_path.exists():
    raise FileNotFoundError(f"Excel não encontrado: {excel_path}")

image_index = index_images_by_name(PASTA_IMAGENS)
if not image_index:
    raise RuntimeError(f"Não foram encontradas imagens em: {PASTA_IMAGENS}")

print(f"[INFO] Excel base: {excel_path}")
print(f"[INFO] Pasta imagens: {PASTA_IMAGENS}")
print(f"[INFO] Pasta saída excels: {output_dir}")
print(f"[INFO] Imagens indexadas: {len(image_index)}")

# cache de tamanhos de imagem
image_size_cache = {}

# workbook que vai sendo degradado cumulativamente
wb = load_workbook(excel_path)

# acumulador por anomalia para permitir forças fracionárias
# chave única por linha: (sheet_name, row)
growth_budget = {}

# ============================================================
# CICLO TEMPORAL
# ============================================================

for step in range(1, NUM_PASSOS + 1):
    months_elapsed = step * MESES_POR_PASSO

    # força deste semestre
    step_strength = get_step_strength(
        step_idx=step,
        total_steps=NUM_PASSOS,
        base_strength=BASE_STEP_STRENGTH,
        decay_rate=DECAY_RATE,
        min_relative_growth=MIN_RELATIVE_GROWTH
    )

    print("\n" + "=" * 68)
    print(f"[PASSO {step:02d}/{NUM_PASSOS}]  Tempo = {months_elapsed} meses")
    print(f"[INFO] Força teórica deste passo = {step_strength:.4f}")
    print("=" * 68)

    total_corrosion_updated = 0
    total_non_corrosion_kept = 0
    total_skipped = 0

    for ws in wb.worksheets:
        print(f"\n--- Folha: {ws.title} ---")

        corrosion_updated_sheet = 0
        non_corrosion_kept_sheet = 0
        skipped_sheet = 0

        for row in range(2, ws.max_row + 1):
            anomaly_type = normalize_anomaly_type(ws[f"C{row}"].value)
            polygon_str = ws[f"E{row}"].value
            area_cm2_old = ws[f"F{row}"].value
            image_path_excel = ws[f"G{row}"].value

            # Se não houver geometria ou imagem associada, ignora
            if not polygon_str or not image_path_excel:
                skipped_sheet += 1
                total_skipped += 1
                continue

            points = parse_polygon_string(polygon_str)
            if len(points) < 3:
                skipped_sheet += 1
                total_skipped += 1
                continue

            # Só corrosão cresce
            if anomaly_type != "corrosion":
                non_corrosion_kept_sheet += 1
                total_non_corrosion_kept += 1
                continue

            image_name = Path(str(image_path_excel)).name.lower()
            size = get_image_size_cached(image_name, image_index, image_size_cache)

            if size is None:
                skipped_sheet += 1
                total_skipped += 1
                continue

            width, height = size

            old_pixel_area = polygon_area_pixels(points)
            old_cm2 = safe_float(area_cm2_old, default=0.0)

            if old_pixel_area <= 0:
                skipped_sheet += 1
                total_skipped += 1
                continue

            row_key = (ws.title, row)
            prev_budget = growth_budget.get(row_key, 0.0)
            new_budget = prev_budget + step_strength

            # quantas iterações inteiras aplicar neste passo
            iterations_to_apply = int(math.floor(new_budget))
            growth_budget[row_key] = new_budget - iterations_to_apply

            # Se neste semestre ainda não atingiu 1 iteração,
            # mantém como está e passa orçamento sobrante ao próximo passo
            if iterations_to_apply <= 0:
                corrosion_updated_sheet += 1
                total_corrosion_updated += 1
                continue

            # 1) polígono -> máscara
            mask = polygon_to_mask(points, width, height)

            # 2) dilatar máscara
            grown_mask = dilate_mask(
                mask,
                kernel_size=KERNEL_SIZE,
                iterations=iterations_to_apply
            )

            # 3) opcional: fechar pequenas falhas
            if APPLY_MORPH_CLOSE:
                grown_mask = close_mask(grown_mask, kernel_size=MORPH_CLOSE_KERNEL_SIZE)

            # 4) máscara -> novo polígono
            new_points = mask_to_polygon(grown_mask)

            if len(new_points) < 3:
                skipped_sheet += 1
                total_skipped += 1
                continue

            new_pixel_area = polygon_area_pixels(new_points)

            # mesma conversão implícita pixel² -> cm² do registo original
            if old_cm2 > 0:
                cm2_per_pixel2 = old_cm2 / old_pixel_area
                new_cm2 = new_pixel_area * cm2_per_pixel2
            else:
                new_cm2 = 0.0

            # atualizar Excel em memória
            ws[f"E{row}"] = polygon_to_string(new_points)
            ws[f"F{row}"] = round(new_cm2, AREA_DECIMALS)

            corrosion_updated_sheet += 1
            total_corrosion_updated += 1

        print(f"Corrosion processadas:      {corrosion_updated_sheet}")
        print(f"Damage/Water mantidas:      {non_corrosion_kept_sheet}")
        print(f"Ignoradas por erro/falta:   {skipped_sheet}")

    # guardar snapshot deste instante temporal
    output_name = build_output_filename(
        base_name=FICHEIRO_EXCEL,
        step_idx=step,
        months_elapsed=months_elapsed
    )
    output_path = output_dir / output_name
    wb.save(output_path)

    print("\n[SNAPSHOT GUARDADO]")
    print(f"Ficheiro: {output_path}")
    print(f"Corrosion processadas: {total_corrosion_updated}")
    print(f"Damage/Water mantidas: {total_non_corrosion_kept}")
    print(f"Ignoradas:             {total_skipped}")

print("\n" + "=" * 68)
print("PROCESSO COMPLETO")
print("=" * 68)
print(f"Excel base: {excel_path}")
print(f"Excels gerados: {NUM_PASSOS}")
print(f"Pasta de saída: {output_dir}")

## Gerar Exceis - c/ tendência

In [ ]:
# ============================================================
# DEGRADAÇÃO TEMPORAL DE CORROSÃO ORIENTADA PELA FORMA ORIGINAL
#
# - Só cresce anomalias cuja coluna C seja "corrosion"
# - "damage" e "water" mantêm-se inalteradas
# - Gera 20 ficheiros Excel, de 6 em 6 meses, durante 10 anos
# - O crescimento desacelera ao longo do tempo
# - Analisa a orientação e o alongamento de cada corrosão
# - Exagera a tendência geométrica da segmentação original
# - Mantém a direção original durante toda a simulação
# ============================================================

import re
import math
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from openpyxl import load_workbook
from PIL import Image


# ============================================================
# CONFIG
# ============================================================

# Pasta onde está o Excel original
PASTA_BASE = r"BASE_EXCEL"
FICHEIRO_EXCEL = "metricas_anomalias_faces_malha.xlsx"

# Pasta das imagens base, para obter largura e altura corretas
PASTA_IMAGENS = r"UNDISTORT_IMAGES"

# Pasta onde serão guardados os 20 ficheiros Excel
PASTA_SAIDA_EXCELS = r"OUTPUTS"


# ------------------------------------------------------------
# PARÂMETROS TEMPORAIS
# ------------------------------------------------------------

NUM_PASSOS = 20
MESES_POR_PASSO = 6


# ------------------------------------------------------------
# INTENSIDADE TEMPORAL DA DEGRADAÇÃO
# ------------------------------------------------------------

# Intensidade inicial acumulada em cada passo.
#
# Valores maiores:
# - crescimento mais rápido;
# - mais iterações de dilatação.
#
# Como os kernels agora podem ser maiores na direção principal,
# não é aconselhável aumentar demasiado este valor.
BASE_STEP_STRENGTH = 1.8

# Controla a desaceleração ao longo do tempo.
# Quanto maior, mais depressa a degradação abranda.
DECAY_RATE = 2.8

# Crescimento relativo mínimo no final da simulação.
# 0.15 significa que, no último passo, ainda existe pelo menos
# 15% da intensidade inicial.
MIN_RELATIVE_GROWTH = 0.15


# ------------------------------------------------------------
# PARÂMETROS DO CRESCIMENTO ORIENTADO
# ------------------------------------------------------------

# Tamanho base do kernel.
#
# Deve ser um número ímpar >= 3.
# Um kernel 3 corresponde a um raio transversal de 1 píxel.
BASE_KERNEL_SIZE = 3

# Número máximo de píxeis adicionais no raio principal.
#
# Exemplo:
# BASE_KERNEL_SIZE = 3 -> raio base = 1
# MAX_EXTRA_MAJOR_RADIUS = 3 -> raio principal máximo = 4
# kernel máximo aproximado = 9 x 9
#
# Aumentar este valor exagera mais a tendência da segmentação.
MAX_EXTRA_MAJOR_RADIUS = 3

# Razão de alongamento a partir da qual se aplica a
# anisotropia máxima.
#
# Exemplo:
# eixo principal / eixo secundário >= 4
# -> crescimento direcional máximo.
ELONGATION_RATIO_FOR_MAX_GROWTH = 4.0

# Controla quanto as tendências moderadas são exageradas.
#
# Valores inferiores a 1:
# - tornam o crescimento direcional mais evidente;
# - mesmo anomalias moderadamente alongadas recebem kernels maiores.
#
# Valores superiores a 1:
# - reservam o crescimento direcional para formas muito alongadas.
TREND_RESPONSE_EXPONENT = 0.70

# Número mínimo de píxeis para considerar a análise PCA fiável.
# Abaixo deste valor, usa-se crescimento aproximadamente uniforme.
MIN_PIXELS_FOR_TREND_ANALYSIS = 25

# Limite apenas para evitar rácios extremos em geometrias
# muito pequenas ou quase lineares.
MAX_ELONGATION_RATIO = 20.0


# ------------------------------------------------------------
# OPÇÕES MORFOLÓGICAS
# ------------------------------------------------------------

# Fechamento morfológico depois da dilatação.
# Ajuda a preencher pequenas aberturas e a deixar a corrosão coesa.
APPLY_MORPH_CLOSE = True
MORPH_CLOSE_KERNEL_SIZE = 3

# Simplificação do polígono extraído da máscara.
#
# Valor menor:
# - mais pontos;
# - maior detalhe;
# - Excel potencialmente maior.
#
# Valor maior:
# - menos pontos;
# - contornos mais simplificados.
POLYGON_SIMPLIFICATION_FACTOR = 0.0015


# ------------------------------------------------------------
# OUTRAS OPÇÕES
# ------------------------------------------------------------

AREA_DECIMALS = 6

EXTENSOES_VALIDAS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp",
    ".webp",
}

# Mostrar no terminal a tendência de cada corrosão.
# Pode produzir muitas linhas quando existem milhares de anomalias.
PRINT_TREND_DETAILS = False


# ============================================================
# ESTRUTURA DE DADOS DA TENDÊNCIA
# ============================================================

@dataclass
class CorrosionTrend:
    """
    Informação calculada a partir da segmentação original.

    A orientação e o alongamento são mantidos durante toda
    a simulação para evitar que a direção mude artificialmente
    entre diferentes instantes temporais.
    """

    angle_deg: float
    elongation_ratio: float
    trend_strength: float

    major_radius: int
    minor_radius: int

    kernel: np.ndarray

    original_pixel_area: float
    original_area_cm2: float
    cm2_per_pixel2: float


# ============================================================
# FUNÇÕES AUXILIARES DE LEITURA E ESCRITA
# ============================================================

def parse_polygon_string(poly_str):
    """
    Converte:

        [100,200]//[150,210]//[170,260]

    numa lista:

        [(100.0, 200.0), (150.0, 210.0), (170.0, 260.0)]
    """

    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()

    if not poly_str:
        return []

    matches = re.findall(
        r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]",
        poly_str,
    )

    return [(float(x), float(y)) for x, y in matches]


def polygon_to_string(points):
    """
    Converte uma lista de pontos no formato guardado no Excel.
    """

    return "//".join(
        f"[{int(round(x))},{int(round(y))}]"
        for x, y in points
    )


def polygon_area_pixels(points):
    """
    Calcula a área do polígono em píxeis quadrados através
    da fórmula de Shoelace.
    """

    if len(points) < 3:
        return 0.0

    pts = np.asarray(points, dtype=np.float64)

    x = pts[:, 0]
    y = pts[:, 1]

    area = 0.5 * abs(
        np.dot(x, np.roll(y, -1))
        - np.dot(y, np.roll(x, -1))
    )

    return float(area)


def safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def normalize_anomaly_type(value):
    if value is None:
        return ""

    return str(value).strip().lower()


# ============================================================
# FUNÇÕES AUXILIARES DAS IMAGENS
# ============================================================

def index_images_by_name(root_folder):
    """
    Indexa recursivamente as imagens pelo nome do ficheiro.
    """

    root = Path(root_folder)
    image_map = {}

    for path in root.rglob("*"):
        if (
            path.is_file()
            and path.suffix.lower() in EXTENSOES_VALIDAS
        ):
            image_map[path.name.lower()] = path

    return image_map


def get_image_size_cached(
    image_name,
    image_index,
    size_cache,
):
    """
    Obtém a largura e altura da imagem, usando cache para evitar
    abrir repetidamente o mesmo ficheiro.
    """

    if image_name in size_cache:
        return size_cache[image_name]

    image_path = image_index.get(image_name)

    if image_path is None:
        return None

    try:
        with Image.open(image_path) as image:
            size_cache[image_name] = image.size

    except Exception as exc:
        print(
            f"[AVISO] Não foi possível abrir a imagem "
            f"{image_path}: {exc}"
        )
        return None

    return size_cache[image_name]


# ============================================================
# FUNÇÕES DE MÁSCARA E CONTORNO
# ============================================================

def polygon_to_mask(points, width, height):
    """
    Converte um polígono numa máscara binária.
    """

    mask = np.zeros((height, width), dtype=np.uint8)

    if len(points) < 3:
        return mask

    clipped_points = []

    for x, y in points:
        x_int = min(
            max(int(round(x)), 0),
            width - 1,
        )

        y_int = min(
            max(int(round(y)), 0),
            height - 1,
        )

        clipped_points.append([x_int, y_int])

    contour = np.asarray(
        clipped_points,
        dtype=np.int32,
    ).reshape((-1, 1, 2))

    cv2.fillPoly(
        mask,
        [contour],
        255,
    )

    return mask


def mask_to_polygon(mask):
    """
    Extrai o maior contorno da máscara e converte-o num polígono.
    """

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )

    if not contours:
        return []

    contour = max(
        contours,
        key=cv2.contourArea,
    )

    perimeter = cv2.arcLength(
        contour,
        True,
    )

    epsilon = POLYGON_SIMPLIFICATION_FACTOR * perimeter

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True,
    )

    points = [
        (
            float(point[0][0]),
            float(point[0][1]),
        )
        for point in approx
    ]

    return points


def close_mask(mask, kernel_size=3):
    """
    Aplica fechamento morfológico leve.
    """

    if kernel_size < 1:
        return mask

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size, kernel_size),
    )

    return cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel,
    )


# ============================================================
# ANÁLISE DA TENDÊNCIA DA SEGMENTAÇÃO
# ============================================================

def validate_configuration():
    """
    Valida os parâmetros que poderiam provocar erros.
    """

    if BASE_KERNEL_SIZE < 3:
        raise ValueError(
            "BASE_KERNEL_SIZE deve ser maior ou igual a 3."
        )

    if BASE_KERNEL_SIZE % 2 == 0:
        raise ValueError(
            "BASE_KERNEL_SIZE deve ser um número ímpar."
        )

    if MAX_EXTRA_MAJOR_RADIUS < 0:
        raise ValueError(
            "MAX_EXTRA_MAJOR_RADIUS não pode ser negativo."
        )

    if ELONGATION_RATIO_FOR_MAX_GROWTH <= 1.0:
        raise ValueError(
            "ELONGATION_RATIO_FOR_MAX_GROWTH deve ser superior a 1."
        )

    if TREND_RESPONSE_EXPONENT <= 0:
        raise ValueError(
            "TREND_RESPONSE_EXPONENT deve ser superior a 0."
        )


def analyse_shape_trend(mask):
    """
    Analisa a forma original através dos momentos geométricos.

    A matriz de covariância da distribuição dos píxeis permite
    obter os dois eixos principais da segmentação:

        eixo principal:
            direção em que a anomalia é mais alongada;

        eixo secundário:
            direção perpendicular ao eixo principal.

    Retorna:
        angle_deg
        elongation_ratio
        trend_strength
    """

    pixel_count = int(
        cv2.countNonZero(mask)
    )

    if pixel_count < MIN_PIXELS_FOR_TREND_ANALYSIS:
        return 0.0, 1.0, 0.0

    moments = cv2.moments(
        mask,
        binaryImage=True,
    )

    m00 = moments["m00"]

    if m00 <= 0:
        return 0.0, 1.0, 0.0

    # Covariância espacial da máscara
    covariance_xx = moments["mu20"] / m00
    covariance_xy = moments["mu11"] / m00
    covariance_yy = moments["mu02"] / m00

    covariance_matrix = np.array(
        [
            [covariance_xx, covariance_xy],
            [covariance_xy, covariance_yy],
        ],
        dtype=np.float64,
    )

    eigenvalues, eigenvectors = np.linalg.eigh(
        covariance_matrix
    )

    order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    major_variance = max(
        float(eigenvalues[0]),
        1e-9,
    )

    minor_variance = max(
        float(eigenvalues[1]),
        1e-9,
    )

    major_vector = eigenvectors[:, 0]

    angle_deg = math.degrees(
        math.atan2(
            major_vector[1],
            major_vector[0],
        )
    )

    # Como se trata de um eixo e não de uma seta,
    # 0° e 180° representam a mesma orientação.
    angle_deg = angle_deg % 180.0

    # Razão entre os comprimentos estimados dos eixos.
    # Como os eigenvalues representam variância, usa-se a raiz.
    elongation_ratio = math.sqrt(
        major_variance / minor_variance
    )

    elongation_ratio = max(
        1.0,
        min(
            elongation_ratio,
            MAX_ELONGATION_RATIO,
        ),
    )

    # Normalização da tendência entre 0 e 1.
    normalized_trend = (
        elongation_ratio - 1.0
    ) / (
        ELONGATION_RATIO_FOR_MAX_GROWTH - 1.0
    )

    normalized_trend = float(
        np.clip(
            normalized_trend,
            0.0,
            1.0,
        )
    )

    # Exagera tendências moderadas quando o expoente é < 1.
    trend_strength = (
        normalized_trend
        ** TREND_RESPONSE_EXPONENT
    )

    return (
        angle_deg,
        elongation_ratio,
        trend_strength,
    )


def create_rotated_elliptical_kernel(
    major_radius,
    minor_radius,
    angle_deg,
):
    """
    Cria um kernel elíptico orientado segundo o eixo principal
    da corrosão.

    O kernel cresce mais na direção do major_radius e menos
    na direção do minor_radius.
    """

    major_radius = max(
        int(major_radius),
        1,
    )

    minor_radius = max(
        int(minor_radius),
        1,
    )

    maximum_radius = max(
        major_radius,
        minor_radius,
    )

    kernel_size = (
        2 * maximum_radius
    ) + 1

    kernel = np.zeros(
        (kernel_size, kernel_size),
        dtype=np.uint8,
    )

    center = (
        maximum_radius,
        maximum_radius,
    )

    cv2.ellipse(
        kernel,
        center=center,
        axes=(
            major_radius,
            minor_radius,
        ),
        angle=float(angle_deg),
        startAngle=0,
        endAngle=360,
        color=1,
        thickness=-1,
    )

    # Garante que o centro pertence ao kernel.
    kernel[
        maximum_radius,
        maximum_radius,
    ] = 1

    return kernel


def create_corrosion_trend(
    original_points,
    width,
    height,
    original_area_cm2,
):
    """
    Calcula e guarda a tendência da segmentação original.
    """

    original_mask = polygon_to_mask(
        original_points,
        width,
        height,
    )

    (
        angle_deg,
        elongation_ratio,
        trend_strength,
    ) = analyse_shape_trend(
        original_mask
    )

    base_radius = (
        BASE_KERNEL_SIZE - 1
    ) // 2

    extra_major_radius = int(
        round(
            MAX_EXTRA_MAJOR_RADIUS
            * trend_strength
        )
    )

    major_radius = (
        base_radius
        + extra_major_radius
    )

    minor_radius = base_radius

    kernel = create_rotated_elliptical_kernel(
        major_radius=major_radius,
        minor_radius=minor_radius,
        angle_deg=angle_deg,
    )

    original_pixel_area = polygon_area_pixels(
        original_points
    )

    if (
        original_pixel_area > 0
        and original_area_cm2 > 0
    ):
        cm2_per_pixel2 = (
            original_area_cm2
            / original_pixel_area
        )
    else:
        cm2_per_pixel2 = 0.0

    return CorrosionTrend(
        angle_deg=angle_deg,
        elongation_ratio=elongation_ratio,
        trend_strength=trend_strength,
        major_radius=major_radius,
        minor_radius=minor_radius,
        kernel=kernel,
        original_pixel_area=original_pixel_area,
        original_area_cm2=original_area_cm2,
        cm2_per_pixel2=cm2_per_pixel2,
    )


def apply_directional_growth(
    mask,
    trend,
    iterations,
):
    """
    Aplica crescimento anisotrópico segundo a tendência original.
    """

    if iterations <= 0:
        return mask.copy()

    grown_mask = cv2.dilate(
        mask,
        trend.kernel,
        iterations=iterations,
    )

    if APPLY_MORPH_CLOSE:
        grown_mask = close_mask(
            grown_mask,
            kernel_size=MORPH_CLOSE_KERNEL_SIZE,
        )

    return grown_mask


# ============================================================
# FUNÇÕES TEMPORAIS
# ============================================================

def get_step_strength(
    step_idx,
    total_steps,
    base_strength,
    decay_rate,
    min_relative_growth,
):
    """
    Calcula a intensidade da degradação para um passo temporal.

    step_idx:
        intervalo entre 1 e total_steps.

    No início:
        intensidade próxima de base_strength.

    No final:
        intensidade reduzida, mas nunca inferior à fração
        definida por min_relative_growth.
    """

    if total_steps <= 1:
        return base_strength

    normalized_time = (
        step_idx - 1
    ) / (
        total_steps - 1
    )

    exponential_decay = math.exp(
        -decay_rate * normalized_time
    )

    relative_strength = (
        min_relative_growth
        + (
            1.0 - min_relative_growth
        ) * exponential_decay
    )

    return (
        base_strength
        * relative_strength
    )


def build_output_filename(
    base_name,
    step_idx,
    months_elapsed,
):
    """
    Gera nomes como:

    metricas_anomalias_faces_malha_degradacao_t01_00a_06m.xlsx
    metricas_anomalias_faces_malha_degradacao_t20_10a_00m.xlsx
    """

    base_path = Path(base_name)

    years = months_elapsed // 12
    remaining_months = months_elapsed % 12

    return (
        f"{base_path.stem}"
        f"_degradacao_t{step_idx:02d}"
        f"_{years:02d}a"
        f"_{remaining_months:02d}m"
        f"{base_path.suffix}"
    )


# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

def main():
    validate_configuration()

    excel_path = (
        Path(PASTA_BASE)
        / FICHEIRO_EXCEL
    )

    output_dir = Path(
        PASTA_SAIDA_EXCELS
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not excel_path.exists():
        raise FileNotFoundError(
            f"Excel não encontrado: {excel_path}"
        )

    image_index = index_images_by_name(
        PASTA_IMAGENS
    )

    if not image_index:
        raise RuntimeError(
            f"Não foram encontradas imagens em: "
            f"{PASTA_IMAGENS}"
        )

    print("=" * 76)
    print("DEGRADAÇÃO TEMPORAL ORIENTADA DA CORROSÃO")
    print("=" * 76)
    print(f"Excel base:          {excel_path}")
    print(f"Pasta de imagens:    {PASTA_IMAGENS}")
    print(f"Pasta de saída:      {output_dir}")
    print(f"Imagens indexadas:   {len(image_index)}")
    print(f"Número de passos:    {NUM_PASSOS}")
    print(f"Meses por passo:     {MESES_POR_PASSO}")
    print("=" * 76)

    # Cache das dimensões das imagens
    image_size_cache = {}

    # Workbook cumulativo
    wb = load_workbook(
        excel_path
    )

    # Acumulador que permite intensidades fracionárias.
    #
    # Exemplo:
    # passo 1 -> 0.7 iterações acumuladas
    # passo 2 -> +0.7 = 1.4
    # aplica 1 iteração e guarda 0.4
    growth_budget = {}

    # Cache da tendência original.
    #
    # A tendência é calculada apenas uma vez, usando a
    # segmentação inicial, e permanece fixa ao longo dos 10 anos.
    trend_cache = {}

    for step in range(
        1,
        NUM_PASSOS + 1,
    ):
        months_elapsed = (
            step
            * MESES_POR_PASSO
        )

        step_strength = get_step_strength(
            step_idx=step,
            total_steps=NUM_PASSOS,
            base_strength=BASE_STEP_STRENGTH,
            decay_rate=DECAY_RATE,
            min_relative_growth=MIN_RELATIVE_GROWTH,
        )

        print("\n" + "=" * 76)
        print(
            f"[PASSO {step:02d}/{NUM_PASSOS}] "
            f"Tempo = {months_elapsed} meses"
        )
        print(
            f"[INFO] Intensidade teórica = "
            f"{step_strength:.4f}"
        )
        print("=" * 76)

        total_corrosion_processed = 0
        total_corrosion_changed = 0
        total_non_corrosion_kept = 0
        total_skipped = 0
        total_iterations = 0

        for worksheet in wb.worksheets:
            print(
                f"\n--- Folha: "
                f"{worksheet.title} ---"
            )

            corrosion_processed_sheet = 0
            corrosion_changed_sheet = 0
            non_corrosion_kept_sheet = 0
            skipped_sheet = 0
            iterations_sheet = 0

            for row in range(
                2,
                worksheet.max_row + 1,
            ):
                anomaly_type = normalize_anomaly_type(
                    worksheet[f"C{row}"].value
                )

                polygon_string = (
                    worksheet[f"E{row}"].value
                )

                area_cm2_value = (
                    worksheet[f"F{row}"].value
                )

                image_path_excel = (
                    worksheet[f"G{row}"].value
                )

                # Sem polígono ou sem imagem associada
                if (
                    not polygon_string
                    or not image_path_excel
                ):
                    skipped_sheet += 1
                    total_skipped += 1
                    continue

                current_points = parse_polygon_string(
                    polygon_string
                )

                if len(current_points) < 3:
                    skipped_sheet += 1
                    total_skipped += 1
                    continue

                # Damage e water não sofrem alterações
                if anomaly_type != "corrosion":
                    non_corrosion_kept_sheet += 1
                    total_non_corrosion_kept += 1
                    continue

                image_name = Path(
                    str(image_path_excel)
                ).name.lower()

                image_size = get_image_size_cached(
                    image_name=image_name,
                    image_index=image_index,
                    size_cache=image_size_cache,
                )

                if image_size is None:
                    print(
                        f"[AVISO] Imagem não encontrada: "
                        f"{image_name}"
                    )

                    skipped_sheet += 1
                    total_skipped += 1
                    continue

                width, height = image_size

                current_pixel_area = polygon_area_pixels(
                    current_points
                )

                if current_pixel_area <= 0:
                    skipped_sheet += 1
                    total_skipped += 1
                    continue

                row_key = (
                    worksheet.title,
                    row,
                )

                # ------------------------------------------------
                # CALCULAR A TENDÊNCIA APENAS NA PRIMEIRA VEZ
                # ------------------------------------------------

                if row_key not in trend_cache:
                    original_area_cm2 = safe_float(
                        area_cm2_value,
                        default=0.0,
                    )

                    trend = create_corrosion_trend(
                        original_points=current_points,
                        width=width,
                        height=height,
                        original_area_cm2=original_area_cm2,
                    )

                    trend_cache[row_key] = trend

                    if PRINT_TREND_DETAILS:
                        print(
                            f"[TENDÊNCIA] "
                            f"Linha={row} | "
                            f"ângulo={trend.angle_deg:.2f}° | "
                            f"alongamento="
                            f"{trend.elongation_ratio:.2f} | "
                            f"intensidade="
                            f"{trend.trend_strength:.3f} | "
                            f"raios="
                            f"{trend.major_radius}/"
                            f"{trend.minor_radius}"
                        )

                trend = trend_cache[row_key]

                # ------------------------------------------------
                # ACUMULAR INTENSIDADE FRACIONÁRIA
                # ------------------------------------------------

                previous_budget = growth_budget.get(
                    row_key,
                    0.0,
                )

                updated_budget = (
                    previous_budget
                    + step_strength
                )

                iterations_to_apply = int(
                    math.floor(
                        updated_budget
                    )
                )

                growth_budget[row_key] = (
                    updated_budget
                    - iterations_to_apply
                )

                corrosion_processed_sheet += 1
                total_corrosion_processed += 1

                if iterations_to_apply <= 0:
                    continue

                # ------------------------------------------------
                # POLÍGONO ATUAL -> MÁSCARA
                # ------------------------------------------------

                current_mask = polygon_to_mask(
                    current_points,
                    width,
                    height,
                )

                # ------------------------------------------------
                # CRESCIMENTO SEGUNDO A TENDÊNCIA ORIGINAL
                # ------------------------------------------------

                grown_mask = apply_directional_growth(
                    mask=current_mask,
                    trend=trend,
                    iterations=iterations_to_apply,
                )

                # ------------------------------------------------
                # MÁSCARA -> NOVO POLÍGONO
                # ------------------------------------------------

                new_points = mask_to_polygon(
                    grown_mask
                )

                if len(new_points) < 3:
                    skipped_sheet += 1
                    total_skipped += 1
                    continue

                new_pixel_area = polygon_area_pixels(
                    new_points
                )

                if new_pixel_area <= 0:
                    skipped_sheet += 1
                    total_skipped += 1
                    continue

                # Utiliza sempre a relação calculada a partir
                # da segmentação original, evitando acumular
                # erros de arredondamento ao longo dos anos.
                if trend.cm2_per_pixel2 > 0:
                    new_area_cm2 = (
                        new_pixel_area
                        * trend.cm2_per_pixel2
                    )
                else:
                    new_area_cm2 = 0.0

                # ------------------------------------------------
                # ATUALIZAR O EXCEL EM MEMÓRIA
                # ------------------------------------------------

                worksheet[f"E{row}"] = polygon_to_string(
                    new_points
                )

                worksheet[f"F{row}"] = round(
                    new_area_cm2,
                    AREA_DECIMALS,
                )

                corrosion_changed_sheet += 1
                total_corrosion_changed += 1

                iterations_sheet += iterations_to_apply
                total_iterations += iterations_to_apply

            print(
                f"Corrosion processadas:       "
                f"{corrosion_processed_sheet}"
            )

            print(
                f"Corrosion alteradas:         "
                f"{corrosion_changed_sheet}"
            )

            print(
                f"Iterações aplicadas:         "
                f"{iterations_sheet}"
            )

            print(
                f"Damage/Water mantidas:       "
                f"{non_corrosion_kept_sheet}"
            )

            print(
                f"Ignoradas por erro/falta:    "
                f"{skipped_sheet}"
            )

        # --------------------------------------------------------
        # GUARDAR O SNAPSHOT DO INSTANTE TEMPORAL
        # --------------------------------------------------------

        output_name = build_output_filename(
            base_name=FICHEIRO_EXCEL,
            step_idx=step,
            months_elapsed=months_elapsed,
        )

        output_path = (
            output_dir
            / output_name
        )

        wb.save(
            output_path
        )

        print("\n[SNAPSHOT GUARDADO]")
        print(f"Ficheiro:               {output_path}")
        print(
            f"Corrosion processadas:  "
            f"{total_corrosion_processed}"
        )
        print(
            f"Corrosion alteradas:    "
            f"{total_corrosion_changed}"
        )
        print(
            f"Iterações aplicadas:    "
            f"{total_iterations}"
        )
        print(
            f"Damage/Water mantidas:  "
            f"{total_non_corrosion_kept}"
        )
        print(
            f"Ignoradas:              "
            f"{total_skipped}"
        )

    print("\n" + "=" * 76)
    print("PROCESSO COMPLETO")
    print("=" * 76)
    print(f"Excel base:       {excel_path}")
    print(f"Excels gerados:   {NUM_PASSOS}")
    print(f"Pasta de saída:   {output_dir}")
    print(
        f"Tendências analisadas: "
        f"{len(trend_cache)}"
    )


if __name__ == "__main__":
    main()

## Reconstruir Imagens

In [ ]:
# ============================================================
# RECONSTRUIR IMAGENS DE TODOS OS "EVENTOS" (TODOS OS EXCELS)
# OUTPUT POR EVENTO:
#   1) overlay
#   2) comparison (original | overlay)
#
# ALTERAÇÃO:
# - Lê o AnomalyID da coluna A
# - Extrai o número depois de "#"
# - Desenha esse número no centróide da anomalia/polígono
# ============================================================

import re
from pathlib import Path
from collections import defaultdict

from openpyxl import load_workbook
from PIL import Image, ImageDraw, ImageFont

# ============================================================
# CONFIG
# ============================================================

# Pasta onde estão os excels temporais gerados
PASTA_EXCELS_EVENTOS = r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\Processing\Exports\Aumento_Ambiental_3D_HIBRIDO_REALISTA"

# Pasta de entrada das imagens base
NOVA_PASTA_IMAGENS = r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\Processing\Imagens_png_pp"

# Pasta de saída global para a reconstrução de imagens dos vários instantes
PASTA_SAIDA = r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\Processing\Exports\Imagens"

# Subpastas por evento
SUBPASTA_OVERLAY = "overlay"
SUBPASTA_COMPARISON = "comparison"

# Processar apenas Excels temporais de degradação.
# Se quiseres mesmo todos os .xlsx, usa "*.xlsx"
EXCEL_GLOB = "*.xlsx"
# EXCEL_GLOB = "*_degradacao_t*.xlsx"

# Extensões aceites
EXTENSOES_VALIDAS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# Se quiseres contorno visível além do preenchimento
DESENHAR_CONTORNO = False
ESPESSURA_CONTORNO = 2

# Espaço entre original e overlay na imagem comparison
COMPARISON_GAP = 20
COR_GAP = (255, 255, 255)

# Cores exatas das anomalias
CORES_ANOMALIA = {
    "corrosion": (255, 0, 0),   # vermelho puro
    "damage":    (0, 255, 0),   # verde puro
    "water":     (0, 0, 255),   # azul puro
}

COR_DEFAULT = (255, 255, 255)

# ------------------------------------------------------------
# IDs DAS ANOMALIAS
# ------------------------------------------------------------
DESENHAR_IDS = True

# Tamanho do texto do ID
FONTE_ID_SIZE = 32

# Se quiseres indicar uma fonte específica, coloca aqui o caminho.
# Exemplo:
# FONTE_ID_PATH = r"C:\Windows\Fonts\arial.ttf"
FONTE_ID_PATH = None

# Cores do texto/caixa
COR_TEXTO_ID = (255, 255, 255)   # branco
COR_FUNDO_ID = (0, 0, 0)         # preto

# Espaço interno da caixa de texto
ID_PADDING = 5

# Se True, desenha uma caixa preta atrás do número
DESENHAR_CAIXA_ID = True


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def parse_polygon_string(poly_str):
    """
    Converte string:
      [x,y]//[x,y]//[x,y]
    para:
      [(x, y), (x, y), ...]
    """
    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()
    if not poly_str:
        return []

    matches = re.findall(
        r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]",
        poly_str
    )

    return [(float(x), float(y)) for x, y in matches]


def clip_points(points, width, height):
    """
    Garante que todos os pontos ficam dentro dos limites da imagem.
    """
    clipped = []

    for x, y in points:
        x = min(max(int(round(x)), 0), width - 1)
        y = min(max(int(round(y)), 0), height - 1)
        clipped.append((x, y))

    return clipped


def get_color(anomaly_type):
    """
    Devolve a cor correspondente ao tipo de anomalia na coluna C.
    """
    if anomaly_type is None:
        return COR_DEFAULT

    key = str(anomaly_type).strip().lower()
    return CORES_ANOMALIA.get(key, COR_DEFAULT)


def extract_id_after_hash(anomaly_id):
    """
    Extrai o número depois de '#'.

    Exemplos:
      Figura_1#0    -> 0
      Figura_100#3  -> 3
      Figura_1050#57 -> 57
    """
    if anomaly_id is None:
        return ""

    s = str(anomaly_id).strip()

    if "#" in s:
        return s.rsplit("#", 1)[1].strip()

    # fallback caso não exista '#'
    return s


def polygon_centroid(points):
    """
    Calcula o centróide geométrico do polígono.

    Se o polígono for degenerado, usa a média dos pontos.
    """
    if not points:
        return None

    if len(points) < 3:
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        return sum(xs) / len(xs), sum(ys) / len(ys)

    area_acc = 0.0
    cx_acc = 0.0
    cy_acc = 0.0

    n = len(points)

    for i in range(n):
        x0, y0 = points[i]
        x1, y1 = points[(i + 1) % n]

        cross = x0 * y1 - x1 * y0
        area_acc += cross
        cx_acc += (x0 + x1) * cross
        cy_acc += (y0 + y1) * cross

    signed_area = area_acc / 2.0

    # Polígono degenerado ou área muito pequena
    if abs(signed_area) < 1e-9:
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        return sum(xs) / len(xs), sum(ys) / len(ys)

    cx = cx_acc / (6.0 * signed_area)
    cy = cy_acc / (6.0 * signed_area)

    return cx, cy


def load_id_font(font_size):
    """
    Carrega a fonte para os IDs.
    Se não encontrar a fonte indicada, tenta Arial.
    Se também falhar, usa a fonte default do PIL.
    """
    if FONTE_ID_PATH:
        try:
            return ImageFont.truetype(FONTE_ID_PATH, font_size)
        except Exception as e:
            print(f"[AVISO] Não foi possível carregar FONTE_ID_PATH: {FONTE_ID_PATH}")
            print(f"        Erro: {e}")

    try:
        return ImageFont.truetype("arial.ttf", font_size)
    except:
        return ImageFont.load_default()


def draw_centered_label(draw, image_size, text, center, font):
    """
    Desenha uma etiqueta centrada num ponto.
    A caixa é ajustada para não sair dos limites da imagem.
    """
    if not text or center is None:
        return

    width, height = image_size
    cx, cy = center

    # Dimensão do texto
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
    except:
        text_w, text_h = draw.textsize(text, font=font)

    box_w = text_w + 2 * ID_PADDING
    box_h = text_h + 2 * ID_PADDING

    x0 = int(round(cx - box_w / 2))
    y0 = int(round(cy - box_h / 2))

    # Garantir que a caixa fica dentro da imagem
    x0 = max(0, min(x0, max(0, width - box_w)))
    y0 = max(0, min(y0, max(0, height - box_h)))

    x1 = x0 + box_w
    y1 = y0 + box_h

    if DESENHAR_CAIXA_ID:
        draw.rectangle([x0, y0, x1, y1], fill=COR_FUNDO_ID)

    draw.text(
        (x0 + ID_PADDING, y0 + ID_PADDING),
        text,
        fill=COR_TEXTO_ID,
        font=font
    )


def index_images_by_name(root_folder):
    """
    Indexa todas as imagens da pasta nova por nome de ficheiro.
    """
    root = Path(root_folder)
    image_map = {}

    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[p.name.lower()] = p

    return image_map


def make_side_by_side(original_img, overlay_img, gap=20, gap_color=(255, 255, 255)):
    """
    Cria uma imagem lado a lado:
      [original] gap [overlay]
    """
    original_img = original_img.convert("RGB")
    overlay_img = overlay_img.convert("RGB")

    w1, h1 = original_img.size
    w2, h2 = overlay_img.size

    out_w = w1 + gap + w2
    out_h = max(h1, h2)

    canvas = Image.new("RGB", (out_w, out_h), gap_color)
    canvas.paste(original_img, (0, 0))
    canvas.paste(overlay_img, (w1 + gap, 0))

    return canvas


def sanitize_stem(name):
    """
    Remove caracteres problemáticos do nome da pasta/evento.
    """
    return re.sub(r'[\\/*?:"<>|]+', "_", name)


def load_excel_grouped_by_image(excel_path):
    """
    Lê um Excel e devolve:
      por_imagem[image_name] = lista de registos/polígonos

    Colunas usadas:
      A -> AnomalyID
      C -> tipo de anomalia
      E -> polígono
      G -> imagem associada
    """
    wb = load_workbook(excel_path, data_only=True)

    por_imagem = defaultdict(list)
    total_rows = 0
    total_valid = 0

    for ws in wb.worksheets:
        for row in range(2, ws.max_row + 1):
            total_rows += 1

            anomaly_id = ws[f"A{row}"].value
            anomaly_type = ws[f"C{row}"].value
            polygon_str = ws[f"E{row}"].value
            image_path_excel = ws[f"G{row}"].value

            if not polygon_str or not image_path_excel:
                continue

            points = parse_polygon_string(polygon_str)
            if len(points) < 3:
                continue

            image_name = Path(str(image_path_excel)).name.lower()

            por_imagem[image_name].append({
                "sheet": ws.title,
                "row": row,
                "anomaly_id": anomaly_id,
                "anomaly_label": extract_id_after_hash(anomaly_id),
                "anomaly_type": anomaly_type,
                "points": points,
            })

            total_valid += 1

    return por_imagem, total_rows, total_valid


def process_one_event_excel(excel_path, image_index, output_root):
    """
    Processa um Excel de evento e gera:
      output_root/<excel_stem>/overlay
      output_root/<excel_stem>/comparison
    """
    event_name = sanitize_stem(excel_path.stem)
    event_dir = output_root / event_name
    overlay_dir = event_dir / SUBPASTA_OVERLAY
    comparison_dir = event_dir / SUBPASTA_COMPARISON

    overlay_dir.mkdir(parents=True, exist_ok=True)
    comparison_dir.mkdir(parents=True, exist_ok=True)

    por_imagem, total_rows, total_valid = load_excel_grouped_by_image(excel_path)

    print("\n" + "=" * 70)
    print(f"[EVENTO] {excel_path.name}")
    print("=" * 70)
    print(f"[INFO] Linhas totais no Excel: {total_rows}")
    print(f"[INFO] Polígonos válidos: {total_valid}")
    print(f"[INFO] Imagens com polígonos: {len(por_imagem)}")

    processadas = 0
    faltam_imagens = 0

    id_font = load_id_font(FONTE_ID_SIZE)

    for image_name, registros in por_imagem.items():
        img_path = image_index.get(image_name)

        if img_path is None:
            print(f"[AVISO] Imagem não encontrada na pasta de imagens: {image_name}")
            faltam_imagens += 1
            continue

        original_img = Image.open(img_path).convert("RGB")

        out_img = original_img.copy()
        w, h = out_img.size
        draw = ImageDraw.Draw(out_img)

        # ----------------------------------------------------
        # 1) DESENHAR PRIMEIRO TODOS OS POLÍGONOS
        # ----------------------------------------------------
        labels_to_draw = []

        for reg in registros:
            pts = clip_points(reg["points"], w, h)

            if len(pts) < 3:
                continue

            cor = get_color(reg["anomaly_type"])

            # Preenchimento sólido da anomalia
            draw.polygon(pts, fill=cor)

            # Contorno opcional
            if DESENHAR_CONTORNO:
                draw.line(pts + [pts[0]], fill=cor, width=ESPESSURA_CONTORNO)

            # Guardar label para desenhar no fim,
            # garantindo que o texto não é tapado por outro polígono
            if DESENHAR_IDS:
                centroid = polygon_centroid(pts)
                label = reg.get("anomaly_label", "")

                labels_to_draw.append({
                    "label": label,
                    "centroid": centroid,
                })

        # ----------------------------------------------------
        # 2) DESENHAR OS IDs NO FIM, POR CIMA DOS POLÍGONOS
        # ----------------------------------------------------
        if DESENHAR_IDS:
            for item in labels_to_draw:
                draw_centered_label(
                    draw=draw,
                    image_size=out_img.size,
                    text=item["label"],
                    center=item["centroid"],
                    font=id_font
                )

        stem = Path(image_name).stem

        # Guardar overlay
        out_overlay = overlay_dir / f"{stem}_overlay.png"
        out_img.save(out_overlay)

        # Guardar comparison
        comparison_img = make_side_by_side(
            original_img=original_img,
            overlay_img=out_img,
            gap=COMPARISON_GAP,
            gap_color=COR_GAP
        )

        out_comparison = comparison_dir / f"{stem}_comparison.png"
        comparison_img.save(out_comparison)

        processadas += 1

    print(f"[OK] Imagens processadas: {processadas}")
    print(f"[OK] Imagens em falta:   {faltam_imagens}")
    print(f"[OK] Overlay em:         {overlay_dir}")
    print(f"[OK] Comparison em:      {comparison_dir}")

    return {
        "event_name": event_name,
        "excel_name": excel_path.name,
        "total_rows": total_rows,
        "total_valid": total_valid,
        "num_images": len(por_imagem),
        "processed": processadas,
        "missing": faltam_imagens,
    }


# ============================================================
# PREPARAR CAMINHOS
# ============================================================

pasta_excels = Path(PASTA_EXCELS_EVENTOS)
nova_pasta_imgs = Path(NOVA_PASTA_IMAGENS)
pasta_saida = Path(PASTA_SAIDA)

if not pasta_excels.exists():
    raise FileNotFoundError(f"Pasta de excels não encontrada: {pasta_excels}")

if not nova_pasta_imgs.exists():
    raise FileNotFoundError(f"Pasta de imagens não encontrada: {nova_pasta_imgs}")

pasta_saida.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Pasta excels eventos: {pasta_excels}")
print(f"[INFO] Pasta imagens:        {nova_pasta_imgs}")
print(f"[INFO] Pasta saída:          {pasta_saida}")
print("[INFO] A indexar imagens...")

image_index = index_images_by_name(nova_pasta_imgs)
print(f"[INFO] Imagens encontradas: {len(image_index)}")

excel_files = sorted(pasta_excels.glob(EXCEL_GLOB))

if not excel_files:
    raise FileNotFoundError(f"Não foram encontrados excels em: {pasta_excels}")

print(f"[INFO] Excels encontrados: {len(excel_files)}")


# ============================================================
# PROCESSAR TODOS OS EVENTOS
# ============================================================

summary = []

for excel_path in excel_files:
    result = process_one_event_excel(
        excel_path=excel_path,
        image_index=image_index,
        output_root=pasta_saida
    )
    summary.append(result)


# ============================================================
# RESUMO FINAL
# ============================================================

print("\n" + "=" * 70)
print("PROCESSAMENTO TOTAL CONCLUÍDO")
print("=" * 70)

total_events = len(summary)
total_processed_imgs = sum(x["processed"] for x in summary)
total_missing_imgs = sum(x["missing"] for x in summary)

print(f"Eventos processados:        {total_events}")
print(f"Total imagens processadas:  {total_processed_imgs}")
print(f"Total imagens em falta:     {total_missing_imgs}")
print(f"Resultados em:              {pasta_saida}")

print("\nResumo por evento:")

for x in summary:
    print(
        f" - {x['excel_name']}: "
        f"processadas={x['processed']}, "
        f"faltam={x['missing']}, "
        f"polígonos_validos={x['total_valid']}"
    )

## Aumento de dados com dados climáticos

In [ ]:
# ============================================================
# AUMENTO DE CORROSÃO 3D CONSTRANGIDO À SUPERFÍCIE DA MESH
# ============================================================
#
# IDEIA:
#   1) O polígono 2D observado é usado para inicializar uma região
#      de vértices 3D sobre a Mesh.ply.
#   2) A corrosão cresce NUM GRAFO 3D LOCAL construído sobre a mesh.
#   3) Mudanças de direção da superfície (quinas, parapeitos, etc.)
#      são consideradas através das normais locais.
#   4) Uma transição de 90° pode ser permitida, mas penalizada.
#   5) A região 3D resultante é projetada novamente para a imagem
#      e gravada em "Coordinates (x,y)".
#   6) "Area (cm2)" passa a representar a área 3D estimada da região
#      sobre a superfície, ancorada à área inicial medida.
#
# IMPORTANTE:
# - NÃO usa rtree.
# - NÃO faz dilatação 2D para decidir o crescimento.
# - A dilatação/expansão ocorre nos vértices 3D da superfície.
# - A projeção 2D é apenas o OUTPUT final.
# - Como a projeção absoluta XML apresentou ~246 px de erro mediano,
#   cada anomalia é ANCORADA ao centróide 2D observado no instante
#   inicial. Assim, o erro de translação absoluto é removido localmente.
#
# REQUISITOS:
#   pip install numpy pandas scipy trimesh openpyxl opencv-python requests
#
# ============================================================

import re
import math
import time
import heapq
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import pandas as pd
import requests
import trimesh

import scipy
from scipy.spatial import cKDTree
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows


# ============================================================
# CONFIGURAÇÃO — ALTERAR APENAS ESTA ZONA
# ============================================================

BASE_EXCEL_PATH = Path(
    r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso"
    r"\Processing\Exports\metricas_anomalias_faces_malha.xlsx"
)

MESH_PATH = Path(
    r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\Mesh.ply"
)

XML_PATH = Path(
    r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\XML.xml"
)

# ------------------------------------------------------------
# AMBIENTE — AUTÓNOMO
# ------------------------------------------------------------
# O código NÃO depende de Excels temporais anteriores.
# Volta a construir internamente o forcing ambiental 2016–2026.

SITE_NAME = "R. dos Trabalhadores do Arco 20, 4780-424 Santo Tirso"
SITE_LAT = 41.33892
SITE_LON = -8.48144

BASE_STATE_DATE = pd.Timestamp("2016-01-01")
SIM_END_DATE = pd.Timestamp("2026-01-01")
STEP_MONTHS = 6

# "open_meteo" -> dados históricos horários
# "csv" -> sensores locais com as colunas indicadas abaixo
ENV_SOURCE = "open_meteo"

SENSOR_CSV = Path(
    r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso"
    r"\sensores_2016_2026.csv"
)

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

# Pesos do índice ambiental relativo.
ENV_WEIGHTS = {
    "time_of_wetness_h": 0.30,
    "precipitation_mm": 0.15,
    "wdr_potential": 0.25,
    "rh_mean": 0.10,
    "wind_p95_ms": 0.10,
    "temperature_factor": 0.10,
}

OUTPUT_DIR = Path(
    r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso"
    r"\Processing\Exports\Aumento_Ambiental_3D_HIBRIDO_REALISTA"
)

# Cache meteorológica independente dos antigos Excels.
WEATHER_CACHE_CSV = OUTPUT_DIR / "weather_hourly_2016_2025.csv"

MASTER_SHEET_NAME = "Anomalies00"
MIRROR_CORROSION_SHEET = "corrosion"

COL_ID = "AnomalyID"
COL_FACADE = "FacadeRoofID"
COL_TYPE = "AnomalyType"
COL_XYZ = "Coordinates"
COL_POLY = "Coordinates (x,y)"
COL_AREA = "Area (cm2)"
COL_IMAGE = "ImagePath"

# ------------------------------------------------------------
# MODO DE TESTE
# ------------------------------------------------------------
# None = processar todas as corrosões válidas.
# Para testar primeiro:
# MAX_ANOMALIES = 30
MAX_ANOMALIES = None

# ------------------------------------------------------------
# CRESCIMENTO
# ------------------------------------------------------------
# Crescimento relativo base da ÁREA 3D por semestre quando o fator
# combinado vale aproximadamente 1.0.
#
# Nesta versão híbrida é propositadamente mais conservador para que
# o contorno aumente de forma gradual e visualmente plausível.
#
# NÃO é uma lei física calibrada de corrosão.
BASE_RELATIVE_AREA_GROWTH_PER_STEP = 0.018

# Limite de segurança para um único semestre.
MAX_RELATIVE_AREA_GROWTH_PER_STEP = 0.060

# Desaceleração temporal semelhante à lógica do código 2D original.
TEMPORAL_DECAY_RATE = 1.8
MIN_TEMPORAL_GROWTH_FACTOR = 0.35

# ------------------------------------------------------------
# PATCH LOCAL 3D
# ------------------------------------------------------------
# O patch é dimensionado automaticamente pela área da corrosão e
# densidade local da mesh.
MIN_LOCAL_VERTICES = 500
MAX_LOCAL_VERTICES = 5000

# Quantas vezes maior que a região inicial deve ser o patch local.
# Deve permitir espaço suficiente para crescimento durante 20 passos.
LOCAL_CAPACITY_FACTOR = 7.0

# Vizinhos usados para construir o grafo local.
GRAPH_K = 12

# Vizinhos usados para estimar as normais via PCA.
NORMAL_K = 18

# Só criamos uma aresta 3D se a distância entre vértices for inferior
# a este fator × espaçamento local mediano.
EDGE_DISTANCE_FACTOR = 2.6

# Evita "saltar" entre duas superfícies paralelas muito próximas.
# Para uma aresta verdadeira sobre a superfície, o vetor entre vértices
# deve ser aproximadamente tangente às duas superfícies.
MAX_EDGE_NORMAL_COMPONENT = 0.90

# ------------------------------------------------------------
# MUDANÇAS DE DIREÇÃO / PARAPEITOS / QUINAS
# ------------------------------------------------------------
# 125° permite acompanhar uma mudança de 90° (parapeito).
# Se quiseres impedir quase totalmente a passagem de quinas:
# usa, por exemplo, 65°.
MAX_NORMAL_TRANSITION_DEG = 125.0

# Penalização progressiva da mudança de normal.
# Em torno de TRANSITION_HALF_DEG, a preferência de crescimento
# já é claramente reduzida.
TRANSITION_HALF_DEG = 50.0
TRANSITION_POWER = 2.0

# ------------------------------------------------------------
# AMBIENTE / PROXIMIDADE
# ------------------------------------------------------------
WATER_RADIUS_M = 3.0
DAMAGE_RADIUS_M = 1.5
CORROSION_RADIUS_M = 1.5

WATER_PROX_WEIGHT = 0.25
DAMAGE_PROX_WEIGHT = 0.12
CORROSION_CLUSTER_WEIGHT = 0.12

# Influência da direção ambiental na ESCOLHA das novas zonas da superfície.
DIRECTION_WEIGHT = 0.45

# Influência da exposição ao vento/chuva na distribuição espacial.
EXPOSURE_WEIGHT = 0.35

# Superfícies mais horizontais e viradas para cima podem reter mais água.
RETENTION_WEIGHT = 0.20

# ------------------------------------------------------------
# PROJEÇÃO
# ------------------------------------------------------------
USE_BROWN_DISTORTION = True

# ------------------------------------------------------------
# CONTORNO 2D REALISTA, MAS CONSTRANGIDO PELO 3D
# ------------------------------------------------------------
# O 3D deixa de ser usado para "desenhar" diretamente o polígono.
# Em vez disso:
#   - o grafo 3D define onde a corrosão pode crescer;
#   - a forma original define o caráter do contorno;
#   - vento/chuva + geometria 3D influenciam a direção;
#   - uma dilatação morfológica pequena produz um contorno coeso.

BASE_KERNEL_SIZE = 3
MAX_EXTRA_MAJOR_RADIUS = 2
ELONGATION_RATIO_FOR_MAX_GROWTH = 4.0
TREND_RESPONSE_EXPONENT = 0.85
MIN_PIXELS_FOR_TREND_ANALYSIS = 25
MAX_ELONGATION_RATIO = 20.0

# Peso da memória da forma original.
# 1.0 -> segue apenas a orientação original.
# 0.0 -> segue apenas a direção projetada do crescimento 3D.
SHAPE_MEMORY_WEIGHT = 0.65

# Em transições fortes (parapeitos/quinas), damos mais peso ao 3D
# para permitir que a direção visual acompanhe a nova superfície.
MAX_GEOMETRY_DIRECTION_WEIGHT = 0.75

# Crescimento morfológico por semestre: mantê-lo pequeno evita
# contornos demasiado inflacionados.
MAX_MORPH_ITERATIONS_PER_STEP = 4

APPLY_MORPH_CLOSE = True
MORPH_CLOSE_KERNEL_SIZE = 3

# A máscara de suporte geométrico inclui a região 3D selecionada
# e alguns anéis adicionais do grafo local.
SURFACE_SUPPORT_RINGS = 3

# Dilatação adicional da máscara de suporte projetada.
SURFACE_SUPPORT_BUFFER_PX = 3

# Se True, o polígono nunca é cortado relativamente ao seu estado anterior.
PRESERVE_PREVIOUS_MASK = True

POLYGON_SIMPLIFICATION_FACTOR = 0.0012

# ------------------------------------------------------------
# CALIBRAÇÃO NORTE
# ------------------------------------------------------------
FACADE_AZIMUTH_DEG = {
    "FN": 0.0,
    "FE": 90.0,
    "FS": 180.0,
    "FO": 270.0,
    "FW": 270.0,
}

NORTH_CALIBRATION_MAX_SAMPLES = 1200

# ------------------------------------------------------------
# OUTPUT / PROGRESSO
# ------------------------------------------------------------
PROGRESS_EVERY = 25
AREA_DECIMALS = 4


# ============================================================
# LOG
# ============================================================

RUN_T0 = time.perf_counter()


def fmt_time(seconds):
    seconds = float(seconds)
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {sec:04.1f}s"
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(minutes):02d}m {sec:04.1f}s"


def status(message):
    elapsed = time.perf_counter() - RUN_T0
    print(f"[{fmt_time(elapsed):>11}] {message}", flush=True)


# ============================================================
# FUNÇÕES BÁSICAS
# ============================================================

def norm_type(value):
    return str(value).strip().lower() if value is not None else ""


def image_stem_any(path_value):
    if path_value is None:
        return ""
    s = str(path_value).strip().replace("\\", "/")
    if not s:
        return ""
    return Path(s.rsplit("/", 1)[-1]).stem.lower()


def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def parse_xyz(value):
    if value is None:
        return None

    nums = re.findall(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?",
        str(value),
    )

    if len(nums) < 3:
        return None

    xyz = np.array(
        [float(nums[0]), float(nums[1]), float(nums[2])],
        dtype=np.float64,
    )

    if not np.all(np.isfinite(xyz)):
        return None

    return xyz


def parse_polygon_string(value):
    if value is None:
        return []

    matches = re.findall(
        r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]",
        str(value),
    )

    return [(float(x), float(y)) for x, y in matches]


def polygon_to_string(points):
    return "//".join(
        f"[{int(round(x))},{int(round(y))}]"
        for x, y in points
    )


def polygon_centroid(points):
    if len(points) < 3:
        if not points:
            return None
        a = np.asarray(points, dtype=float)
        return a.mean(axis=0)

    cnt = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
    m = cv2.moments(cnt)

    if abs(m["m00"]) < 1e-12:
        return np.asarray(points, dtype=float).mean(axis=0)

    return np.array(
        [m["m10"] / m["m00"], m["m01"] / m["m00"]],
        dtype=float,
    )


def angle_between_deg(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)

    if na < 1e-12 or nb < 1e-12:
        return 180.0

    c = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
    return float(np.degrees(np.arccos(c)))


def angular_difference_deg(a, b):
    return abs((float(a) - float(b) + 180.0) % 360.0 - 180.0)


def azimuth_from_xy_vector(v):
    """
    0° = +Y (Norte)
    90° = +X (Este)
    """
    v = np.asarray(v, dtype=float)
    return float(np.degrees(np.arctan2(v[0], v[1])) % 360.0)


def unit_vector_from_azimuth_deg(azimuth_deg):
    """
    Vetor horizontal em coordenadas onde:
      0° -> +Y
      90° -> +X
    """
    a = np.radians(float(azimuth_deg))
    return np.array(
        [np.sin(a), np.cos(a), 0.0],
        dtype=np.float64,
    )


def circular_mean_deg(values):
    if len(values) == 0:
        return 0.0

    a = np.radians(np.asarray(values, dtype=float))
    x = np.mean(np.cos(a))
    y = np.mean(np.sin(a))

    return float(np.degrees(np.arctan2(y, x)) % 360.0)


# ============================================================
# CÂMARAS XML
# ============================================================

@dataclass
class Camera:
    stem: str
    width: int
    height: int
    focal: float
    cx: float
    cy: float
    k1: float
    k2: float
    k3: float
    p1: float
    p2: float
    R: np.ndarray
    C: np.ndarray


def xml_float(parent, path, default=0.0):
    node = parent.find(path)
    if node is None or node.text is None:
        return float(default)
    return float(node.text)


def load_cameras(xml_path):
    root = ET.parse(xml_path).getroot()
    cameras = {}

    for photo in root.findall(".//Photo"):
        image_path = photo.findtext("ImagePath", default="")
        stem = image_stem_any(image_path)

        if not stem:
            continue

        width = int(xml_float(photo, "./ImageDimensions/Width"))
        height = int(xml_float(photo, "./ImageDimensions/Height"))
        focal = xml_float(photo, "./FocalLength")

        cx = xml_float(photo, "./PrincipalPoint/x", width / 2)
        cy = xml_float(photo, "./PrincipalPoint/y", height / 2)

        k1 = xml_float(photo, "./Distortion/k1")
        k2 = xml_float(photo, "./Distortion/k2")
        k3 = xml_float(photo, "./Distortion/k3")
        p1 = xml_float(photo, "./Distortion/p1")
        p2 = xml_float(photo, "./Distortion/p2")

        R = np.array([
            [
                xml_float(photo, f"./Rotation/M_{i}{j}")
                for j in range(3)
            ]
            for i in range(3)
        ], dtype=np.float64)

        C = np.array([
            xml_float(photo, "./Center/x"),
            xml_float(photo, "./Center/y"),
            xml_float(photo, "./Center/z"),
        ], dtype=np.float64)

        cameras[stem] = Camera(
            stem=stem,
            width=width,
            height=height,
            focal=focal,
            cx=cx,
            cy=cy,
            k1=k1,
            k2=k2,
            k3=k3,
            p1=p1,
            p2=p2,
            R=R,
            C=C,
        )

    return cameras


def project_points(points_3d, cam):
    """
    Projeção vetorizada:
        pc = R @ (X - C)

    Usa Brown-Conrady forward distortion porque, nos testes anteriores,
    reduziu o erro de projeção relativamente à projeção sem distorção.
    """
    P = np.asarray(points_3d, dtype=np.float64).reshape(-1, 3)

    d = P - cam.C[None, :]
    pc = d @ cam.R.T

    z = pc[:, 2]
    valid = np.isfinite(z) & (z > 1e-9)

    uv = np.full((len(P), 2), np.nan, dtype=np.float64)

    if not np.any(valid):
        return uv, valid

    x = pc[valid, 0] / z[valid]
    y = pc[valid, 1] / z[valid]

    if USE_BROWN_DISTORTION:
        r2 = x * x + y * y
        r4 = r2 * r2
        r6 = r4 * r2

        radial = (
            1.0
            + cam.k1 * r2
            + cam.k2 * r4
            + cam.k3 * r6
        )

        xd = (
            x * radial
            + 2.0 * cam.p1 * x * y
            + cam.p2 * (r2 + 2.0 * x * x)
        )

        yd = (
            y * radial
            + cam.p1 * (r2 + 2.0 * y * y)
            + 2.0 * cam.p2 * x * y
        )
    else:
        xd = x
        yd = y

    uv[valid, 0] = cam.cx + cam.focal * xd
    uv[valid, 1] = cam.cy + cam.focal * yd

    valid &= np.all(np.isfinite(uv), axis=1)

    return uv, valid


# ============================================================
# ENVIRONMENTAL FORCING — GERADO INTERNAMENTE
# ============================================================

EXPECTED_ENV_COLUMNS = [
    "timestamp",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
]


def robust_ratio(value, reference):
    """
    Razão robusta limitada para impedir que um semestre extremo
    domine completamente o índice ambiental.
    """
    ref = max(float(reference), 1e-9)
    return float(
        np.clip(
            float(value) / ref,
            0.35,
            2.50,
        )
    )


def build_periods():
    starts = pd.date_range(
        BASE_STATE_DATE,
        SIM_END_DATE,
        freq=f"{STEP_MONTHS}MS",
        inclusive="left",
    )

    periods = []

    for start in starts:
        end = start + pd.DateOffset(
            months=STEP_MONTHS
        )

        periods.append(
            (
                pd.Timestamp(start),
                min(
                    pd.Timestamp(end),
                    SIM_END_DATE,
                ),
            )
        )

    return periods


PERIODS = build_periods()


def circular_weighted_mean_deg(direction_deg, weights):
    """
    Média circular ponderada:
      0° = Norte
      90° = Este
    """
    d = np.asarray(
        direction_deg,
        dtype=float,
    )

    w = np.asarray(
        weights,
        dtype=float,
    )

    valid = (
        np.isfinite(d)
        & np.isfinite(w)
        & (w > 0)
    )

    if not np.any(valid):
        return np.nan

    rad = np.deg2rad(
        d[valid]
    )

    ww = w[valid]

    x = np.sum(
        ww * np.sin(rad)
    )

    y = np.sum(
        ww * np.cos(rad)
    )

    if abs(x) + abs(y) < 1e-12:
        return np.nan

    return float(
        np.rad2deg(
            np.arctan2(
                x,
                y,
            )
        ) % 360.0
    )


def temperature_corrosion_factor(temp_c):
    """
    Modulador empírico para aumento de dados.
    NÃO é uma lei calibrada de corrosão.
    """
    t = float(temp_c)

    if t <= 0:
        return 0.55

    if t <= 20:
        return (
            0.55
            + 0.45 * (t / 20.0)
        )

    if t <= 30:
        return 1.0

    return max(
        0.70,
        1.0 - 0.03 * (t - 30.0),
    )


def download_open_meteo_hourly(
    start_date,
    end_date,
):
    """
    Descarrega os dados horários por ano.
    O intervalo usado internamente é [start_date, end_date).
    """
    chunks = []

    for year in range(
        start_date.year,
        end_date.year + 1,
    ):
        s = max(
            pd.Timestamp(
                f"{year}-01-01"
            ),
            start_date,
        )

        e = min(
            pd.Timestamp(
                f"{year}-12-31 23:00"
            ),
            end_date
            - pd.Timedelta(
                hours=1
            ),
        )

        if e < s:
            continue

        params = {
            "latitude": SITE_LAT,
            "longitude": SITE_LON,
            "start_date": s.strftime(
                "%Y-%m-%d"
            ),
            "end_date": e.strftime(
                "%Y-%m-%d"
            ),
            "hourly": ",".join([
                "temperature_2m",
                "relative_humidity_2m",
                "precipitation",
                "wind_speed_10m",
                "wind_direction_10m",
                "shortwave_radiation",
            ]),
            "timezone": "Europe/Lisbon",
            "wind_speed_unit": "ms",
        }

        status(
            f"[METEO] {year}: "
            f"{params['start_date']} -> "
            f"{params['end_date']}"
        )

        response = requests.get(
            OPEN_METEO_URL,
            params=params,
            timeout=120,
        )

        response.raise_for_status()

        payload = response.json()

        if "hourly" not in payload:
            raise RuntimeError(
                f"Resposta Open-Meteo sem 'hourly': "
                f"{payload}"
            )

        frame = pd.DataFrame(
            payload["hourly"]
        )

        if frame.empty:
            raise RuntimeError(
                f"Open-Meteo devolveu dados vazios para {year}."
            )

        frame["timestamp"] = pd.to_datetime(
            frame["time"]
        )

        frame = frame.drop(
            columns=["time"]
        )

        chunks.append(
            frame
        )

    if not chunks:
        raise RuntimeError(
            "Não foram obtidos dados meteorológicos."
        )

    return (
        pd.concat(
            chunks,
            ignore_index=True,
        )
        .sort_values(
            "timestamp"
        )
        .drop_duplicates(
            "timestamp"
        )
        .reset_index(
            drop=True
        )
    )


def load_environment_hourly():
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    if ENV_SOURCE.lower() == "csv":
        if not SENSOR_CSV.exists():
            raise FileNotFoundError(
                f"CSV de sensores não encontrado: "
                f"{SENSOR_CSV}"
            )

        df = pd.read_csv(
            SENSOR_CSV
        )

        missing = [
            c
            for c in EXPECTED_ENV_COLUMNS
            if c not in df.columns
        ]

        if missing:
            raise ValueError(
                f"Faltam colunas no CSV de sensores: "
                f"{missing}"
            )

        df["timestamp"] = pd.to_datetime(
            df["timestamp"]
        )

        return df[
            EXPECTED_ENV_COLUMNS
        ].copy()

    if WEATHER_CACHE_CSV.exists():
        status(
            f"[METEO] A usar cache: "
            f"{WEATHER_CACHE_CSV}"
        )

        return pd.read_csv(
            WEATHER_CACHE_CSV,
            parse_dates=["timestamp"],
        )

    status(
        "[METEO] Cache não encontrada. "
        "A descarregar dados horários do Open-Meteo..."
    )

    df = download_open_meteo_hourly(
        BASE_STATE_DATE,
        SIM_END_DATE,
    )

    df.to_csv(
        WEATHER_CACHE_CSV,
        index=False,
    )

    status(
        f"[METEO] Cache guardada: "
        f"{WEATHER_CACHE_CSV}"
    )

    return df


def aggregate_environment(hourly):
    hourly = hourly.copy()

    hourly["timestamp"] = pd.to_datetime(
        hourly["timestamp"]
    )

    # Time of wetness proxy:
    # RH >= 80% e T > 0°C.
    hourly["wet_hour"] = (
        (
            hourly["relative_humidity_2m"]
            >= 80.0
        )
        & (
            hourly["temperature_2m"]
            > 0.0
        )
    ).astype(float)

    # Índice relativo de chuva dirigida:
    # precipitação × velocidade do vento.
    hourly["wdr_hour"] = (
        hourly["precipitation"]
        .clip(lower=0)
        * hourly["wind_speed_10m"]
        .clip(lower=0)
    )

    rows = []

    for i, (
        start,
        end,
    ) in enumerate(
        PERIODS,
        start=1,
    ):
        p = hourly[
            (
                hourly["timestamp"]
                >= start
            )
            & (
                hourly["timestamp"]
                < end
            )
        ].copy()

        if p.empty:
            raise RuntimeError(
                f"Sem dados ambientais para "
                f"{start} -> {end}"
            )

        rain_w = (
            p["precipitation"]
            .clip(lower=0)
            * p["wind_speed_10m"]
            .clip(lower=0)
        )

        wind_dir = circular_weighted_mean_deg(
            p["wind_direction_10m"],
            p["wind_speed_10m"]
            .clip(lower=0.05),
        )

        wdr_dir = circular_weighted_mean_deg(
            p["wind_direction_10m"],
            rain_w.clip(lower=0),
        )

        if not np.isfinite(wind_dir):
            wind_dir = 0.0

        if not np.isfinite(wdr_dir):
            wdr_dir = wind_dir

        temp_mean = float(
            p["temperature_2m"].mean()
        )

        rows.append({
            "step": i,
            "start": start,
            "end": end,
            "temperature_mean_c": temp_mean,
            "temperature_factor":
                temperature_corrosion_factor(
                    temp_mean
                ),
            "rh_mean":
                float(
                    p["relative_humidity_2m"]
                    .mean()
                ),
            "time_of_wetness_h":
                float(
                    p["wet_hour"]
                    .sum()
                ),
            "precipitation_mm":
                float(
                    p["precipitation"]
                    .sum()
                ),
            "wind_mean_ms":
                float(
                    p["wind_speed_10m"]
                    .mean()
                ),
            "wind_p95_ms":
                float(
                    p["wind_speed_10m"]
                    .quantile(0.95)
                ),
            "wind_direction_deg":
                float(wind_dir),
            "wdr_potential":
                float(
                    p["wdr_hour"]
                    .sum()
                ),
            "wdr_direction_deg":
                float(wdr_dir),
            "shortwave_mean_wm2":
                float(
                    p["shortwave_radiation"]
                    .mean()
                ),
        })

    env = pd.DataFrame(
        rows
    )

    ref_cols = [
        "time_of_wetness_h",
        "precipitation_mm",
        "wdr_potential",
        "rh_mean",
        "wind_p95_ms",
        "temperature_factor",
    ]

    refs = {
        c: float(
            env[c].median()
        )
        for c in ref_cols
    }

    env[
        "environment_factor"
    ] = 0.0

    for column, weight in (
        ENV_WEIGHTS.items()
    ):
        env[
            "environment_factor"
        ] += (
            float(weight)
            * env[column].apply(
                lambda value, c=column:
                    robust_ratio(
                        value,
                        refs[c],
                    )
            )
        )

    return env


def load_environment_schedule():
    """
    Cria os 20 semestres diretamente a partir dos dados meteorológicos.
    Não depende de nenhum Excel temporal anterior.
    """
    hourly = load_environment_hourly()

    status(
        f"[AMBIENTE] Dados horários disponíveis: "
        f"{len(hourly):,}"
    )

    env = aggregate_environment(
        hourly
    )

    return env, WEATHER_CACHE_CSV


# ============================================================
# MESH + NORMAIS
# ============================================================

status("[1/10] A validar caminhos...")

for p in [
    BASE_EXCEL_PATH,
    MESH_PATH,
    XML_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(p)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


status("[2/10] A carregar câmaras XML...")
CAMERAS = load_cameras(XML_PATH)
status(f"[2/10] Câmaras: {len(CAMERAS):,}")


status("[3/10] A ler Excel master...")
MASTER_DF = pd.read_excel(
    BASE_EXCEL_PATH,
    sheet_name=MASTER_SHEET_NAME,
)

MASTER_DF["_excel_row"] = np.arange(2, len(MASTER_DF) + 2)

status(
    f"[3/10] Master: {len(MASTER_DF):,} linhas."
)


status("[4/10] A carregar a mesh...")
_t = time.perf_counter()

mesh = trimesh.load(
    MESH_PATH,
    force="mesh",
    process=False,
)

if not isinstance(mesh, trimesh.Trimesh):
    raise TypeError(
        f"O ficheiro não foi interpretado como Trimesh: {type(mesh)}"
    )

MESH_VERTICES = np.asarray(
    mesh.vertices,
    dtype=np.float64,
)

status(
    f"[4/10] Mesh: {len(MESH_VERTICES):,} vértices | "
    f"{len(mesh.faces):,} faces | "
    f"{fmt_time(time.perf_counter() - _t)}"
)


status("[4/10] A construir KD-tree global da mesh...")
_t = time.perf_counter()

MESH_TREE = cKDTree(MESH_VERTICES)

status(
    f"[4/10] KD-tree global pronta em "
    f"{fmt_time(time.perf_counter() - _t)}."
)


def batch_pca_normals(points, neighbor_indices, source_points):
    """
    PCA vetorizada.
    neighbor_indices: N x K
    """
    neigh = source_points[neighbor_indices]
    center = neigh.mean(axis=1, keepdims=True)
    q = neigh - center

    cov = np.einsum(
        "nki,nkj->nij",
        q,
        q,
        optimize=True,
    )

    denom = max(neigh.shape[1] - 1, 1)
    cov /= float(denom)

    eigvals, eigvecs = np.linalg.eigh(cov)

    normals = eigvecs[:, :, 0]

    norms = np.linalg.norm(
        normals,
        axis=1,
        keepdims=True,
    )

    norms[norms < 1e-12] = 1.0
    normals = normals / norms

    return normals


def orient_normals_toward_camera(normals, points, cam):
    if cam is None:
        return normals

    n = np.asarray(normals, dtype=float).copy()
    p = np.asarray(points, dtype=float)

    to_cam = cam.C[None, :] - p

    flip = np.einsum(
        "ij,ij->i",
        n,
        to_cam,
    ) < 0.0

    n[flip] *= -1.0

    return n


def surface_normals_at_queries(query_points, cameras=None):
    """
    Normal PCA local em torno de cada query usando a KD-tree global.
    """
    Q = np.asarray(query_points, dtype=np.float64).reshape(-1, 3)

    k = min(NORMAL_K, len(MESH_VERTICES))

    distances, ids = MESH_TREE.query(
        Q,
        k=k,
    )

    ids = np.atleast_2d(ids)

    normals = batch_pca_normals(
        Q,
        ids,
        MESH_VERTICES,
    )

    nearest_points = MESH_VERTICES[ids[:, 0]]

    if cameras is not None:
        for i, cam in enumerate(cameras):
            if cam is None:
                continue

            if np.dot(
                normals[i],
                cam.C - nearest_points[i],
            ) < 0:
                normals[i] *= -1.0

    return nearest_points, normals


# ============================================================
# CALIBRAR NORTE
# ============================================================

def estimate_north_offset(df):
    valid_rows = []

    for _, row in df.iterrows():
        facade = str(
            row.get(COL_FACADE, "")
        ).strip().upper()

        if facade not in FACADE_AZIMUTH_DEG:
            continue

        X = parse_xyz(row.get(COL_XYZ))

        if X is None:
            continue

        cam = CAMERAS.get(
            image_stem_any(row.get(COL_IMAGE))
        )

        if cam is None:
            continue

        valid_rows.append(
            (X, cam, FACADE_AZIMUTH_DEG[facade])
        )

        if len(valid_rows) >= NORTH_CALIBRATION_MAX_SAMPLES:
            break

    if not valid_rows:
        status("[NORTE] Sem amostras válidas; offset=0°.")
        return 0.0, np.nan

    Q = np.vstack([x[0] for x in valid_rows])
    cams = [x[1] for x in valid_rows]
    targets = np.array([x[2] for x in valid_rows], dtype=float)

    _, normals = surface_normals_at_queries(
        Q,
        cameras=cams,
    )

    offsets = []
    used_mesh_az = []
    used_targets = []

    for n, target in zip(normals, targets):
        # Calibração por fachadas predominantemente verticais.
        if abs(float(n[2])) > 0.75:
            continue

        h = n.copy()
        h[2] = 0.0

        if np.linalg.norm(h) < 1e-9:
            continue

        mesh_az = azimuth_from_xy_vector(h)

        delta = (
            float(target)
            - mesh_az
        ) % 360.0

        offsets.append(delta)
        used_mesh_az.append(mesh_az)
        used_targets.append(float(target))

    if not offsets:
        status("[NORTE] Não foi possível calibrar; offset=0°.")
        return 0.0, np.nan

    offset = circular_mean_deg(offsets)

    residuals = [
        angular_difference_deg(
            (mesh_az + offset) % 360.0,
            target,
        )
        for mesh_az, target
        in zip(used_mesh_az, used_targets)
    ]

    median_error = float(
        np.median(residuals)
    )

    status(
        f"[NORTE] Offset XY(mesh)->Norte: "
        f"{offset:.2f}° | erro mediano={median_error:.2f}° | "
        f"n={len(residuals)}"
    )

    return offset, median_error


status("[5/10] A calibrar orientação da mesh...")
NORTH_OFFSET_DEG, NORTH_ERROR_DEG = estimate_north_offset(
    MASTER_DF
)


def real_azimuth_from_normal(normal):
    h = np.asarray(normal, dtype=float).copy()
    h[2] = 0.0

    if np.linalg.norm(h) < 1e-12:
        return np.nan

    mesh_az = azimuth_from_xy_vector(h)

    return (
        mesh_az
        + NORTH_OFFSET_DEG
    ) % 360.0


def real_azimuth_to_mesh_vector(azimuth_deg):
    """
    Converte azimute real para vetor horizontal no referencial XY da mesh.
    """
    mesh_az = (
        float(azimuth_deg)
        - NORTH_OFFSET_DEG
    ) % 360.0

    return unit_vector_from_azimuth_deg(
        mesh_az
    )


# ============================================================
# PROXIMIDADE 3D
# ============================================================

status("[6/10] A construir índices de anomalias 3D...")

xyz_by_type = {}

for anomaly_type in [
    "water",
    "damage",
    "corrosion",
]:
    pts = []

    subset = MASTER_DF[
        MASTER_DF[COL_TYPE]
        .astype(str)
        .str.strip()
        .str.lower()
        == anomaly_type
    ]

    for value in subset[COL_XYZ]:
        X = parse_xyz(value)
        if X is not None:
            pts.append(X)

    arr = (
        np.vstack(pts)
        if pts
        else np.empty((0, 3), dtype=float)
    )

    xyz_by_type[anomaly_type] = arr

    status(
        f"[6/10] {anomaly_type}: {len(arr):,} centroides."
    )

ANOMALY_TREES = {
    key: (
        cKDTree(value)
        if len(value)
        else None
    )
    for key, value in xyz_by_type.items()
}


def proximity_factor(X):
    factor = 1.0

    water_tree = ANOMALY_TREES["water"]
    damage_tree = ANOMALY_TREES["damage"]
    corrosion_tree = ANOMALY_TREES["corrosion"]

    d_water = np.inf
    d_damage = np.inf
    n_corr = 0

    if water_tree is not None:
        d_water, _ = water_tree.query(X, k=1)

        if d_water <= WATER_RADIUS_M:
            factor += (
                WATER_PROX_WEIGHT
                * (1.0 - d_water / WATER_RADIUS_M)
            )

    if damage_tree is not None:
        d_damage, _ = damage_tree.query(X, k=1)

        if d_damage <= DAMAGE_RADIUS_M:
            factor += (
                DAMAGE_PROX_WEIGHT
                * (1.0 - d_damage / DAMAGE_RADIUS_M)
            )

    if corrosion_tree is not None:
        ids = corrosion_tree.query_ball_point(
            X,
            r=CORROSION_RADIUS_M,
        )

        # Excluir a própria observação aproximada.
        n_corr = max(len(ids) - 1, 0)

        factor += (
            CORROSION_CLUSTER_WEIGHT
            * min(n_corr / 5.0, 1.0)
        )

    return {
        "factor": float(factor),
        "distance_water_m": float(d_water),
        "distance_damage_m": float(d_damage),
        "corrosion_neighbors": int(n_corr),
    }


# ============================================================
# ENVIRONMENTAL FORCING
# ============================================================

status("[7/10] A construir forcing ambiental 2016–2026...")
ENVIRONMENT, ENVIRONMENT_SOURCE = load_environment_schedule()

status(
    f"[7/10] Passos ambientais: {len(ENVIRONMENT)} | "
    f"{ENVIRONMENT['start'].iloc[0].date()} -> "
    f"{ENVIRONMENT['end'].iloc[-1].date()}"
)


# ============================================================
# FORMA 2D ORIGINAL — MEMÓRIA MORFOLÓGICA
# ============================================================

@dataclass
class ShapeTrend:
    angle_deg: float
    elongation_ratio: float
    trend_strength: float
    original_pixel_area: float


def polygon_area_pixels(points):
    if len(points) < 3:
        return 0.0

    pts = np.asarray(points, dtype=np.float64)
    x = pts[:, 0]
    y = pts[:, 1]

    return float(
        0.5
        * abs(
            np.dot(x, np.roll(y, -1))
            - np.dot(y, np.roll(x, -1))
        )
    )


def polygon_to_local_mask(points, margin=8):
    """
    Converte o polígono numa máscara LOCAL.
    Evita criar sempre uma imagem 5280x3956.
    """
    if len(points) < 3:
        return None, None

    pts = np.asarray(points, dtype=np.float64)

    x0 = int(math.floor(np.min(pts[:, 0]))) - margin
    y0 = int(math.floor(np.min(pts[:, 1]))) - margin
    x1 = int(math.ceil(np.max(pts[:, 0]))) + margin
    y1 = int(math.ceil(np.max(pts[:, 1]))) + margin

    x0 = max(x0, 0)
    y0 = max(y0, 0)

    if x1 <= x0 or y1 <= y0:
        return None, None

    local = pts.copy()
    local[:, 0] -= x0
    local[:, 1] -= y0

    width = max(2, x1 - x0 + 1)
    height = max(2, y1 - y0 + 1)

    mask = np.zeros(
        (height, width),
        dtype=np.uint8,
    )

    contour = np.round(local).astype(
        np.int32
    ).reshape(-1, 1, 2)

    cv2.fillPoly(
        mask,
        [contour],
        255,
    )

    return mask, (x0, y0)


def analyse_polygon_shape(points):
    """
    Analisa a orientação e o alongamento da segmentação original,
    seguindo a lógica do código 2D que produzia formas mais naturais.
    """
    original_area = polygon_area_pixels(points)

    mask, _ = polygon_to_local_mask(
        points,
        margin=3,
    )

    if mask is None:
        return ShapeTrend(
            angle_deg=0.0,
            elongation_ratio=1.0,
            trend_strength=0.0,
            original_pixel_area=max(original_area, 1.0),
        )

    pixel_count = int(
        cv2.countNonZero(mask)
    )

    if pixel_count < MIN_PIXELS_FOR_TREND_ANALYSIS:
        return ShapeTrend(
            angle_deg=0.0,
            elongation_ratio=1.0,
            trend_strength=0.0,
            original_pixel_area=max(original_area, 1.0),
        )

    m = cv2.moments(
        mask,
        binaryImage=True,
    )

    if m["m00"] <= 0:
        return ShapeTrend(
            angle_deg=0.0,
            elongation_ratio=1.0,
            trend_strength=0.0,
            original_pixel_area=max(original_area, 1.0),
        )

    cov = np.array([
        [
            m["mu20"] / m["m00"],
            m["mu11"] / m["m00"],
        ],
        [
            m["mu11"] / m["m00"],
            m["mu02"] / m["m00"],
        ],
    ], dtype=np.float64)

    eigvals, eigvecs = np.linalg.eigh(
        cov
    )

    order = np.argsort(
        eigvals
    )[::-1]

    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    major_var = max(
        float(eigvals[0]),
        1e-9,
    )

    minor_var = max(
        float(eigvals[1]),
        1e-9,
    )

    major_vector = eigvecs[:, 0]

    angle_deg = (
        math.degrees(
            math.atan2(
                major_vector[1],
                major_vector[0],
            )
        )
        % 180.0
    )

    elongation = math.sqrt(
        major_var / minor_var
    )

    elongation = float(
        np.clip(
            elongation,
            1.0,
            MAX_ELONGATION_RATIO,
        )
    )

    normalized = (
        elongation - 1.0
    ) / (
        ELONGATION_RATIO_FOR_MAX_GROWTH - 1.0
    )

    normalized = float(
        np.clip(
            normalized,
            0.0,
            1.0,
        )
    )

    trend_strength = float(
        normalized
        ** TREND_RESPONSE_EXPONENT
    )

    return ShapeTrend(
        angle_deg=float(angle_deg),
        elongation_ratio=float(elongation),
        trend_strength=float(trend_strength),
        original_pixel_area=max(float(original_area), 1.0),
    )


def axial_blend_angle_deg(angle_a, angle_b, weight_b):
    """
    Mistura dois EIXOS (0° == 180°).
    """
    if angle_b is None or not np.isfinite(angle_b):
        return float(angle_a) % 180.0

    w = float(
        np.clip(
            weight_b,
            0.0,
            1.0,
        )
    )

    a = math.radians(
        2.0 * float(angle_a)
    )

    b = math.radians(
        2.0 * float(angle_b)
    )

    x = (
        (1.0 - w) * math.cos(a)
        + w * math.cos(b)
    )

    y = (
        (1.0 - w) * math.sin(a)
        + w * math.sin(b)
    )

    if abs(x) + abs(y) < 1e-12:
        return float(angle_a) % 180.0

    return (
        0.5
        * math.degrees(
            math.atan2(
                y,
                x,
            )
        )
    ) % 180.0


def create_realistic_kernel(
    angle_deg,
    trend_strength,
    exposure,
):
    """
    Kernel pequeno e elíptico.
    Mantém a irregularidade do polígono anterior em vez de o reconstruir.
    """
    base_radius = (
        BASE_KERNEL_SIZE - 1
    ) // 2

    combined_directionality = float(
        np.clip(
            0.75 * trend_strength
            + 0.25 * exposure,
            0.0,
            1.0,
        )
    )

    extra_major = int(
        round(
            MAX_EXTRA_MAJOR_RADIUS
            * combined_directionality
        )
    )

    major_radius = max(
        1,
        base_radius + extra_major,
    )

    minor_radius = max(
        1,
        base_radius,
    )

    max_radius = max(
        major_radius,
        minor_radius,
    )

    size = (
        2 * max_radius
        + 1
    )

    kernel = np.zeros(
        (size, size),
        dtype=np.uint8,
    )

    center = (
        max_radius,
        max_radius,
    )

    cv2.ellipse(
        kernel,
        center=center,
        axes=(
            major_radius,
            minor_radius,
        ),
        angle=float(angle_deg),
        startAngle=0,
        endAngle=360,
        color=1,
        thickness=-1,
    )

    kernel[
        max_radius,
        max_radius,
    ] = 1

    return kernel


def temporal_growth_factor(step_number, total_steps):
    """
    Desaceleração temporal progressiva.
    """
    if total_steps <= 1:
        return 1.0

    t = (
        float(step_number - 1)
        / float(total_steps - 1)
    )

    decay = math.exp(
        -TEMPORAL_DECAY_RATE * t
    )

    return float(
        MIN_TEMPORAL_GROWTH_FACTOR
        + (
            1.0
            - MIN_TEMPORAL_GROWTH_FACTOR
        ) * decay
    )


# ============================================================
# PATCH 3D LOCAL
# ============================================================

@dataclass
class SurfacePatch:
    row_number: int
    anomaly_id: str
    camera: Camera
    points: np.ndarray
    global_vertex_ids: np.ndarray
    normals: np.ndarray
    graph: list
    point_area_m2: np.ndarray
    selected: set
    anchor_uv: np.ndarray
    original_polygon: list
    current_polygon: list
    shape_trend: ShapeTrend
    initial_area_m2: float
    proximity: dict
    area_budget_m2: float
    spacing_m: float


def local_normal_transition_factor(angle_deg):
    if angle_deg > MAX_NORMAL_TRANSITION_DEG:
        return 0.0

    x = (
        float(angle_deg)
        / max(TRANSITION_HALF_DEG, 1e-6)
    )

    return 1.0 / (
        1.0
        + x ** TRANSITION_POWER
    )


def build_local_graph(points, normals):
    n = len(points)

    if n < 2:
        return [[] for _ in range(n)], 0.0, np.ones(n) * 1e-6

    k = min(
        GRAPH_K + 1,
        n,
    )

    tree = cKDTree(points)

    distances, ids = tree.query(
        points,
        k=k,
    )

    if k == 1:
        distances = distances[:, None]
        ids = ids[:, None]

    nn = distances[:, 1]

    finite_nn = nn[
        np.isfinite(nn)
        & (nn > 1e-12)
    ]

    spacing = (
        float(np.median(finite_nn))
        if len(finite_nn)
        else 0.01
    )

    max_edge = (
        EDGE_DISTANCE_FACTOR
        * spacing
    )

    graph_sets = [
        set()
        for _ in range(n)
    ]

    for i in range(n):
        for col in range(1, ids.shape[1]):
            j = int(ids[i, col])

            if j == i:
                continue

            dist = float(distances[i, col])

            if (
                not np.isfinite(dist)
                or dist <= 1e-12
                or dist > max_edge
            ):
                continue

            angle = angle_between_deg(
                normals[i],
                normals[j],
            )

            if angle > MAX_NORMAL_TRANSITION_DEG:
                continue

            edge = (
                points[j]
                - points[i]
            ) / dist

            # Evita ponte direta entre superfícies paralelas próximas.
            normal_component = max(
                abs(float(np.dot(edge, normals[i]))),
                abs(float(np.dot(edge, normals[j]))),
            )

            if normal_component > MAX_EDGE_NORMAL_COMPONENT:
                continue

            graph_sets[i].add(j)
            graph_sets[j].add(i)

    graph = [
        sorted(list(s))
        for s in graph_sets
    ]

    # Área de Voronoi aproximada de malha triangular quase regular.
    # Depois é calibrada para a área inicial real da anomalia.
    point_area_raw = (
        (math.sqrt(3.0) / 2.0)
        * np.maximum(nn, spacing * 0.25) ** 2
    )

    finite_area = point_area_raw[
        np.isfinite(point_area_raw)
        & (point_area_raw > 0)
    ]

    if len(finite_area):
        lo = np.percentile(finite_area, 10)
        hi = np.percentile(finite_area, 90)

        point_area_raw = np.clip(
            point_area_raw,
            max(lo * 0.5, 1e-12),
            max(hi * 2.0, 1e-12),
        )

    return graph, spacing, point_area_raw


def connected_component_inside(graph, inside_ids, start_id):
    allowed = set(int(i) for i in inside_ids)

    if not allowed:
        return set()

    if start_id not in allowed:
        start_id = min(
            allowed,
            key=lambda i: i,
        )

    visited = {int(start_id)}
    q = deque([int(start_id)])

    while q:
        i = q.popleft()

        for j in graph[i]:
            if j in allowed and j not in visited:
                visited.add(j)
                q.append(j)

    return visited


def geodesic_seed(graph, points, point_area_raw, seed_index, target_area_raw):
    """
    Fallback: crescimento geodésico a partir do vértice mais próximo
    até atingir aproximadamente a área alvo.
    """
    selected = set()
    visited = set()

    heap = [(0.0, int(seed_index))]
    accumulated = 0.0

    while heap and accumulated < target_area_raw:
        dist, i = heapq.heappop(heap)

        if i in visited:
            continue

        visited.add(i)
        selected.add(i)

        accumulated += float(
            point_area_raw[i]
        )

        for j in graph[i]:
            if j in visited:
                continue

            edge_len = float(
                np.linalg.norm(
                    points[j]
                    - points[i]
                )
            )

            heapq.heappush(
                heap,
                (
                    dist + edge_len,
                    int(j),
                ),
            )

    return selected


def build_surface_patch(row):
    X = parse_xyz(row.get(COL_XYZ))
    poly = parse_polygon_string(row.get(COL_POLY))
    area_cm2 = safe_float(row.get(COL_AREA), np.nan)

    if (
        X is None
        or len(poly) < 3
        or not np.isfinite(area_cm2)
        or area_cm2 <= 0
    ):
        raise ValueError("XYZ/polígono/área inicial inválido.")

    cam = CAMERAS.get(
        image_stem_any(
            row.get(COL_IMAGE)
        )
    )

    if cam is None:
        raise ValueError("Câmara não encontrada no XML.")

    area_m2 = (
        float(area_cm2)
        / 10000.0
    )

    # --------------------------------------------------------
    # Estimar densidade local da mesh.
    # --------------------------------------------------------
    k_probe = min(
        32,
        len(MESH_VERTICES),
    )

    probe_dist, _ = MESH_TREE.query(
        X,
        k=k_probe,
    )

    probe_dist = np.atleast_1d(
        probe_dist
    )

    positive = probe_dist[
        np.isfinite(probe_dist)
        & (probe_dist > 1e-12)
    ]

    spacing_est = (
        float(np.median(positive[: min(len(positive), 20)]))
        if len(positive)
        else 0.01
    )

    area_per_vertex_est = (
        (math.sqrt(3.0) / 2.0)
        * spacing_est ** 2
    )

    seed_vertex_est = max(
        area_m2
        / max(area_per_vertex_est, 1e-10),
        20.0,
    )

    local_k = int(
        np.clip(
            math.ceil(
                seed_vertex_est
                * LOCAL_CAPACITY_FACTOR
            ),
            MIN_LOCAL_VERTICES,
            MAX_LOCAL_VERTICES,
        )
    )

    local_k = min(
        local_k,
        len(MESH_VERTICES),
    )

    distances, global_ids = MESH_TREE.query(
        X,
        k=local_k,
    )

    global_ids = np.atleast_1d(
        global_ids
    ).astype(np.int64)

    local_points = MESH_VERTICES[
        global_ids
    ].copy()

    # --------------------------------------------------------
    # Normais locais por PCA dentro do próprio patch.
    # --------------------------------------------------------
    local_tree = cKDTree(
        local_points
    )

    nk = min(
        NORMAL_K,
        len(local_points),
    )

    _, normal_ids = local_tree.query(
        local_points,
        k=nk,
    )

    if nk == 1:
        normal_ids = normal_ids[:, None]

    local_normals = batch_pca_normals(
        local_points,
        normal_ids,
        local_points,
    )

    local_normals = orient_normals_toward_camera(
        local_normals,
        local_points,
        cam,
    )

    # --------------------------------------------------------
    # Grafo 3D da superfície.
    # --------------------------------------------------------
    graph, spacing, point_area_raw = build_local_graph(
        local_points,
        local_normals,
    )

    # --------------------------------------------------------
    # Âncora 2D:
    # corrigir localmente o erro absoluto da projeção da câmara.
    # --------------------------------------------------------
    poly_centroid = polygon_centroid(poly)

    uv_X, ok_X = project_points(
        X.reshape(1, 3),
        cam,
    )

    if not ok_X[0]:
        raise ValueError(
            "O centróide 3D não é projetável nesta câmara."
        )

    anchor_uv = (
        np.asarray(poly_centroid, dtype=float)
        - uv_X[0]
    )

    # --------------------------------------------------------
    # Projetar o patch local e usar o polígono observado para
    # identificar a região 3D inicial.
    # --------------------------------------------------------
    uv_local, ok = project_points(
        local_points,
        cam,
    )

    uv_local = (
        uv_local
        + anchor_uv[None, :]
    )

    contour = np.asarray(
        poly,
        dtype=np.float32,
    ).reshape(-1, 1, 2)

    inside = []

    for i in range(len(local_points)):
        if not ok[i]:
            continue

        u, v = uv_local[i]

        if not (
            np.isfinite(u)
            and np.isfinite(v)
        ):
            continue

        if cv2.pointPolygonTest(
            contour,
            (float(u), float(v)),
            False,
        ) >= 0:
            inside.append(i)

    # --------------------------------------------------------
    # Escolher o componente 3D conectado correspondente.
    # --------------------------------------------------------
    nearest_local = int(
        np.argmin(
            np.linalg.norm(
                local_points - X[None, :],
                axis=1,
            )
        )
    )

    if inside:
        start_inside = min(
            inside,
            key=lambda i: float(
                np.linalg.norm(
                    local_points[i]
                    - X
                )
            ),
        )

        selected = connected_component_inside(
            graph,
            inside,
            start_inside,
        )
    else:
        selected = set()

    # --------------------------------------------------------
    # Fallback se a projeção não capturar vértices suficientes.
    # --------------------------------------------------------
    if len(selected) < 3:
        target_area_raw = max(
            area_m2,
            float(
                np.median(point_area_raw)
            ) * 5.0,
        )

        selected = geodesic_seed(
            graph,
            local_points,
            point_area_raw,
            nearest_local,
            target_area_raw,
        )

    if len(selected) < 1:
        raise ValueError(
            "Não foi possível inicializar região 3D conectada."
        )

    # --------------------------------------------------------
    # Calibrar a área por vértice para que o estado inicial 3D
    # reproduza EXATAMENTE a área inicial do Excel.
    # --------------------------------------------------------
    seed_ids = np.array(
        sorted(selected),
        dtype=int,
    )

    raw_seed_area = float(
        np.sum(
            point_area_raw[
                seed_ids
            ]
        )
    )

    if raw_seed_area <= 1e-12:
        raise ValueError(
            "Área local estimada degenerada."
        )

    area_scale = (
        area_m2
        / raw_seed_area
    )

    point_area_m2 = (
        point_area_raw
        * area_scale
    )

    prox = proximity_factor(X)

    shape_trend = analyse_polygon_shape(
        poly
    )

    return SurfacePatch(
        row_number=int(row["_excel_row"]),
        anomaly_id=str(row.get(COL_ID)),
        camera=cam,
        points=local_points,
        global_vertex_ids=global_ids,
        normals=local_normals,
        graph=graph,
        point_area_m2=point_area_m2,
        selected=set(selected),
        anchor_uv=anchor_uv,
        original_polygon=list(poly),
        current_polygon=list(poly),
        shape_trend=shape_trend,
        initial_area_m2=float(area_m2),
        proximity=prox,
        area_budget_m2=0.0,
        spacing_m=float(spacing),
    )


# ============================================================
# CRESCIMENTO 3D
# ============================================================

def candidate_score(
    patch,
    parent,
    candidate,
    env_row,
):
    p0 = patch.points[parent]
    p1 = patch.points[candidate]

    n0 = patch.normals[parent]
    n1 = patch.normals[candidate]

    edge = p1 - p0
    edge_norm = np.linalg.norm(edge)

    if edge_norm < 1e-12:
        return 0.0, 0.0, 180.0

    edge /= edge_norm

    transition_angle = angle_between_deg(
        n0,
        n1,
    )

    transition = local_normal_transition_factor(
        transition_angle
    )

    if transition <= 0:
        return 0.0, 0.0, transition_angle

    # --------------------------------------------------------
    # Direção da chuva/vento:
    # wdr_direction é assumida como direção DE ONDE vem.
    # --------------------------------------------------------
    source_real_az = float(
        env_row["wdr_direction_deg"]
    )

    source_vec = real_azimuth_to_mesh_vector(
        source_real_az
    )

    # Vetor para onde se desloca vento/chuva.
    travel_vec = -source_vec

    # --------------------------------------------------------
    # Exposição da superfície.
    # Normal horizontal apontando para o exterior.
    # --------------------------------------------------------
    horizontal = n1.copy()
    horizontal[2] = 0.0

    nh = np.linalg.norm(horizontal)

    if nh > 1e-12:
        horizontal /= nh

        exposure = max(
            0.0,
            float(
                np.dot(
                    horizontal,
                    source_vec,
                )
            ),
        )
    else:
        # Superfície horizontal: sem "fachada" diretamente exposta,
        # mas continua suscetível a precipitação/retensão.
        exposure = 0.35

    # --------------------------------------------------------
    # Direção preferencial tangencial na superfície.
    # --------------------------------------------------------
    tangent = (
        travel_vec
        - np.dot(
            travel_vec,
            n1,
        ) * n1
    )

    nt = np.linalg.norm(tangent)

    if nt > 1e-12:
        tangent /= nt

        directional = max(
            0.0,
            float(
                np.dot(
                    edge,
                    tangent,
                )
            ),
        )
    else:
        directional = 0.5

    # --------------------------------------------------------
    # Retenção:
    # superfícies viradas para cima recebem maior peso.
    # --------------------------------------------------------
    retention = (
        0.5
        + 0.5
        * max(
            0.0,
            float(n1[2]),
        )
    )

    score = transition

    score *= (
        (1.0 - EXPOSURE_WEIGHT)
        + EXPOSURE_WEIGHT * exposure
    )

    score *= (
        (1.0 - DIRECTION_WEIGHT)
        + DIRECTION_WEIGHT * directional
    )

    score *= (
        (1.0 - RETENTION_WEIGHT)
        + RETENTION_WEIGHT * retention
    )

    return (
        float(score),
        float(exposure),
        float(transition_angle),
    )


def get_frontier_candidates(patch, env_row):
    selected = patch.selected

    candidates = {}

    for parent in selected:
        for candidate in patch.graph[parent]:
            if candidate in selected:
                continue

            score, exposure, transition_angle = candidate_score(
                patch,
                parent,
                candidate,
                env_row,
            )

            if score <= 0:
                continue

            previous = candidates.get(
                candidate
            )

            item = {
                "candidate": int(candidate),
                "parent": int(parent),
                "score": float(score),
                "exposure": float(exposure),
                "transition_angle": float(transition_angle),
            }

            if (
                previous is None
                or item["score"] > previous["score"]
            ):
                candidates[candidate] = item

    return list(
        candidates.values()
    )


def grow_patch_one_step(
    patch,
    env_row,
):
    selected_ids = np.array(
        sorted(patch.selected),
        dtype=int,
    )

    current_area = float(
        np.sum(
            patch.point_area_m2[
                selected_ids
            ]
        )
    )

    frontier = get_frontier_candidates(
        patch,
        env_row,
    )

    if frontier:
        mean_exposure = float(
            np.mean([
                x["exposure"]
                for x in frontier
            ])
        )
    else:
        mean_exposure = 0.0

    env_factor = float(
        env_row["environment_factor"]
    )

    prox_factor = float(
        patch.proximity["factor"]
    )

    # A geometria controla também a magnitude total,
    # mas de forma moderada.
    geometry_magnitude = (
        0.80
        + 0.20 * mean_exposure
    )

    temporal_factor = temporal_growth_factor(
        int(env_row["step"]),
        len(ENVIRONMENT),
    )

    relative_growth = (
        BASE_RELATIVE_AREA_GROWTH_PER_STEP
        * env_factor
        * prox_factor
        * geometry_magnitude
        * temporal_factor
    )

    relative_growth = float(
        np.clip(
            relative_growth,
            0.0,
            MAX_RELATIVE_AREA_GROWTH_PER_STEP,
        )
    )

    target_delta = (
        current_area
        * relative_growth
    )

    patch.area_budget_m2 += (
        target_delta
    )

    added_area = 0.0
    added_vertices = 0
    transitions_used = []
    added_vertex_ids = []
    added_parent_ids = []

    # --------------------------------------------------------
    # Consumir orçamento adicionando vértices do frontier.
    # Se o crescimento atravessar uma quina, os candidatos da nova
    # superfície aparecem naturalmente no frontier seguinte.
    # --------------------------------------------------------
    safety = 0

    while patch.area_budget_m2 > 0:
        safety += 1

        if safety > len(patch.points) + 10:
            break

        frontier = get_frontier_candidates(
            patch,
            env_row,
        )

        if not frontier:
            break

        # Melhor candidato primeiro.
        frontier.sort(
            key=lambda x: x["score"],
            reverse=True,
        )

        chosen = None

        # Preferir candidato cuja área cabe no orçamento.
        for item in frontier:
            area_i = float(
                patch.point_area_m2[
                    item["candidate"]
                ]
            )

            if area_i <= patch.area_budget_m2:
                chosen = item
                break

        # Se nenhum cabe, acumular orçamento para o próximo semestre.
        if chosen is None:
            break

        cid = chosen["candidate"]

        patch.selected.add(
            cid
        )

        area_i = float(
            patch.point_area_m2[
                cid
            ]
        )

        patch.area_budget_m2 -= area_i

        added_area += area_i
        added_vertices += 1

        transitions_used.append(
            chosen["transition_angle"]
        )

        added_vertex_ids.append(
            int(cid)
        )
        added_parent_ids.append(
            int(chosen["parent"])
        )

        if len(patch.selected) >= len(patch.points):
            break

    new_ids = np.array(
        sorted(patch.selected),
        dtype=int,
    )

    new_area = float(
        np.sum(
            patch.point_area_m2[
                new_ids
            ]
        )
    )

    return {
        "area_before_m2": current_area,
        "area_after_m2": new_area,
        "added_area_m2": added_area,
        "added_vertices": int(added_vertices),
        "relative_growth_target": float(relative_growth),
        "temporal_growth_factor": float(temporal_factor),
        "mean_frontier_exposure": float(mean_exposure),
        "added_vertex_ids": added_vertex_ids,
        "added_parent_ids": added_parent_ids,
        "max_transition_used_deg": (
            float(max(transitions_used))
            if transitions_used
            else 0.0
        ),
        "mean_transition_used_deg": (
            float(np.mean(transitions_used))
            if transitions_used
            else 0.0
        ),
    }


# ============================================================
# 3D -> DIREÇÃO / SUPORTE -> CONTORNO 2D REALISTA
# ============================================================

def projected_growth_axis_deg(
    patch,
    growth,
):
    """
    Calcula a orientação 2D das arestas 3D realmente adicionadas.
    Assim, ao atravessar um parapeito, a direção projetada pode mudar.
    """
    child_ids = growth.get(
        "added_vertex_ids",
        [],
    )

    parent_ids = growth.get(
        "added_parent_ids",
        [],
    )

    if not child_ids:
        return None

    angles = []
    weights = []

    for parent, child in zip(
        parent_ids,
        child_ids,
    ):
        points = np.vstack([
            patch.points[parent],
            patch.points[child],
        ])

        uv, valid = project_points(
            points,
            patch.camera,
        )

        if not np.all(valid):
            continue

        uv += patch.anchor_uv[
            None,
            :
        ]

        d = (
            uv[1]
            - uv[0]
        )

        length = float(
            np.linalg.norm(d)
        )

        if length < 1e-6:
            continue

        angle = (
            math.degrees(
                math.atan2(
                    d[1],
                    d[0],
                )
            )
            % 180.0
        )

        angles.append(
            angle
        )

        weights.append(
            length
        )

    if not angles:
        return None

    # Média axial ponderada.
    a = np.radians(
        2.0
        * np.asarray(
            angles,
            dtype=float,
        )
    )

    w = np.asarray(
        weights,
        dtype=float,
    )

    x = np.sum(
        w * np.cos(a)
    )

    y = np.sum(
        w * np.sin(a)
    )

    if abs(x) + abs(y) < 1e-12:
        return None

    return (
        0.5
        * math.degrees(
            math.atan2(
                y,
                x,
            )
        )
    ) % 180.0


def collect_surface_support_ids(
    patch,
):
    """
    Região 3D atualmente ocupada + alguns anéis topológicos.
    Não inclui superfícies sem conectividade no grafo.
    """
    support = set(
        patch.selected
    )

    frontier = set(
        patch.selected
    )

    for _ in range(
        SURFACE_SUPPORT_RINGS
    ):
        nxt = set()

        for i in frontier:
            for j in patch.graph[i]:
                if j not in support:
                    nxt.add(
                        int(j)
                    )

        if not nxt:
            break

        support.update(
            nxt
        )

        frontier = nxt

    return sorted(
        support
    )


def projected_support_points(
    patch,
):
    ids = collect_surface_support_ids(
        patch
    )

    if not ids:
        return np.empty(
            (0, 2),
            dtype=float,
        )

    P = patch.points[
        np.asarray(
            ids,
            dtype=int,
        )
    ]

    uv, valid = project_points(
        P,
        patch.camera,
    )

    uv += patch.anchor_uv[
        None,
        :
    ]

    uv = uv[
        valid
    ]

    finite = np.all(
        np.isfinite(uv),
        axis=1,
    )

    return uv[
        finite
    ]


def polygon_from_local_mask(
    mask,
    x0,
    y0,
):
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )

    if not contours:
        return []

    contour = max(
        contours,
        key=cv2.contourArea,
    )

    if len(contour) < 3:
        return []

    perimeter = cv2.arcLength(
        contour,
        True,
    )

    epsilon = (
        POLYGON_SIMPLIFICATION_FACTOR
        * perimeter
    )

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True,
    )

    result = []

    for p in approx:
        u = float(
            p[0][0]
            + x0
        )

        v = float(
            p[0][1]
            + y0
        )

        u = float(
            np.clip(
                u,
                0,
                patch_camera_width_global - 1,
            )
        )

        v = float(
            np.clip(
                v,
                0,
                patch_camera_height_global - 1,
            )
        )

        result.append(
            (u, v)
        )

    return result


def realistic_polygon_from_3d(
    patch,
    growth,
):
    """
    O 3D determina:
      - área física alvo;
      - superfície permitida;
      - direção dominante;
      - mudança de direção em quinas/parapeitos.

    A morfologia 2D determina:
      - continuidade;
      - irregularidade;
      - preservação da forma original;
      - contorno visualmente plausível.
    """
    global patch_camera_width_global
    global patch_camera_height_global

    patch_camera_width_global = (
        patch.camera.width
    )

    patch_camera_height_global = (
        patch.camera.height
    )

    current_poly = list(
        patch.current_polygon
    )

    if len(current_poly) < 3:
        current_poly = list(
            patch.original_polygon
        )

    current_pixel_area = polygon_area_pixels(
        current_poly
    )

    if current_pixel_area <= 0:
        return (
            current_poly,
            patch.shape_trend.angle_deg,
            0.0,
            0,
        )

    # A área 3D define apenas QUANTO o contorno deve aumentar.
    area_ratio = (
        float(
            growth["area_after_m2"]
        )
        / max(
            patch.initial_area_m2,
            1e-12,
        )
    )

    target_pixel_area = (
        patch.shape_trend.original_pixel_area
        * area_ratio
    )

    if target_pixel_area <= (
        current_pixel_area
        * 1.001
    ):
        return (
            current_poly,
            patch.shape_trend.angle_deg,
            1.0,
            0,
        )

    # --------------------------------------------------------
    # Direção:
    # memória da forma original + direção real do crescimento 3D.
    # --------------------------------------------------------
    geometry_angle = projected_growth_axis_deg(
        patch,
        growth,
    )

    transition = float(
        growth.get(
            "max_transition_used_deg",
            0.0,
        )
    )

    exposure = float(
        growth.get(
            "mean_frontier_exposure",
            0.0,
        )
    )

    geometry_weight = (
        1.0
        - SHAPE_MEMORY_WEIGHT
    )

    # Numa mudança forte de superfície, permitir que a geometria
    # assuma mais controlo da direção.
    geometry_weight += (
        0.35
        * np.clip(
            transition / 90.0,
            0.0,
            1.0,
        )
    )

    geometry_weight += (
        0.15
        * np.clip(
            exposure,
            0.0,
            1.0,
        )
    )

    geometry_weight = float(
        np.clip(
            geometry_weight,
            0.0,
            MAX_GEOMETRY_DIRECTION_WEIGHT,
        )
    )

    final_angle = axial_blend_angle_deg(
        patch.shape_trend.angle_deg,
        geometry_angle,
        geometry_weight,
    )

    kernel = create_realistic_kernel(
        angle_deg=final_angle,
        trend_strength=patch.shape_trend.trend_strength,
        exposure=exposure,
    )

    # --------------------------------------------------------
    # Suporte geométrico projetado.
    # --------------------------------------------------------
    support_uv = projected_support_points(
        patch
    )

    pts = np.asarray(
        current_poly,
        dtype=float,
    )

    all_x = list(
        pts[:, 0]
    )
    all_y = list(
        pts[:, 1]
    )

    if len(support_uv):
        all_x.extend(
            support_uv[:, 0].tolist()
        )
        all_y.extend(
            support_uv[:, 1].tolist()
        )

    kernel_radius = (
        max(
            kernel.shape
        )
        // 2
    )

    margin = (
        12
        + kernel_radius
        * MAX_MORPH_ITERATIONS_PER_STEP
        + SURFACE_SUPPORT_BUFFER_PX
    )

    x0 = max(
        0,
        int(
            math.floor(
                min(all_x)
                - margin
            )
        ),
    )

    y0 = max(
        0,
        int(
            math.floor(
                min(all_y)
                - margin
            )
        ),
    )

    x1 = min(
        patch.camera.width - 1,
        int(
            math.ceil(
                max(all_x)
                + margin
            )
        ),
    )

    y1 = min(
        patch.camera.height - 1,
        int(
            math.ceil(
                max(all_y)
                + margin
            )
        ),
    )

    if x1 <= x0 or y1 <= y0:
        return (
            current_poly,
            final_angle,
            0.0,
            0,
        )

    W = (
        x1 - x0 + 1
    )

    H = (
        y1 - y0 + 1
    )

    current_mask = np.zeros(
        (H, W),
        dtype=np.uint8,
    )

    local_current = np.asarray(
        [
            [
                int(round(x - x0)),
                int(round(y - y0)),
            ]
            for x, y in current_poly
        ],
        dtype=np.int32,
    ).reshape(-1, 1, 2)

    cv2.fillPoly(
        current_mask,
        [local_current],
        255,
    )

    # --------------------------------------------------------
    # Criar máscara suave da superfície 3D alcançável.
    # --------------------------------------------------------
    support_mask = np.zeros(
        (H, W),
        dtype=np.uint8,
    )

    valid_support = 0

    if len(support_uv):
        local_support = (
            support_uv
            - np.array(
                [x0, y0],
                dtype=float,
            )[None, :]
        )

        keep = (
            (local_support[:, 0] >= 0)
            & (local_support[:, 0] < W)
            & (local_support[:, 1] >= 0)
            & (local_support[:, 1] < H)
        )

        local_support = local_support[
            keep
        ]

        valid_support = len(
            local_support
        )

        if valid_support:
            # Espaçamento projetado para escolher raio adequado.
            if valid_support >= 4:
                tree2 = cKDTree(
                    local_support
                )

                d2, _ = tree2.query(
                    local_support,
                    k=2,
                )

                spacing_px = float(
                    np.median(
                        d2[:, 1][
                            np.isfinite(
                                d2[:, 1]
                            )
                        ]
                    )
                )
            else:
                spacing_px = 2.0

            support_radius = int(
                np.clip(
                    math.ceil(
                        max(
                            1.0,
                            0.85 * spacing_px,
                        )
                    )
                    + SURFACE_SUPPORT_BUFFER_PX,
                    2,
                    12,
                )
            )

            for u, v in local_support:
                cv2.circle(
                    support_mask,
                    (
                        int(round(u)),
                        int(round(v)),
                    ),
                    support_radius,
                    255,
                    -1,
                )

            # Suavizar sem transformar a geometria numa mancha convexa.
            support_kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,
                (3, 3),
            )

            support_mask = cv2.morphologyEx(
                support_mask,
                cv2.MORPH_CLOSE,
                support_kernel,
                iterations=1,
            )

    # A forma atual deve permanecer sempre válida.
    support_mask = cv2.bitwise_or(
        support_mask,
        current_mask,
    )

    # --------------------------------------------------------
    # Crescimento progressivo, no máximo alguns pixels por semestre.
    # --------------------------------------------------------
    mask = current_mask.copy()

    iterations_used = 0

    for _ in range(
        MAX_MORPH_ITERATIONS_PER_STEP
    ):
        before = int(
            cv2.countNonZero(
                mask
            )
        )

        if before >= target_pixel_area:
            break

        proposal = cv2.dilate(
            mask,
            kernel,
            iterations=1,
        )

        # A expansão só é aceite onde a superfície 3D conectada permite.
        proposal = cv2.bitwise_and(
            proposal,
            support_mask,
        )

        if PRESERVE_PREVIOUS_MASK:
            proposal = cv2.bitwise_or(
                proposal,
                mask,
            )

        if APPLY_MORPH_CLOSE:
            close_kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,
                (
                    MORPH_CLOSE_KERNEL_SIZE,
                    MORPH_CLOSE_KERNEL_SIZE,
                ),
            )

            proposal = cv2.morphologyEx(
                proposal,
                cv2.MORPH_CLOSE,
                close_kernel,
            )

            # O fecho não pode criar pixels fora da superfície permitida.
            proposal = cv2.bitwise_and(
                proposal,
                cv2.bitwise_or(
                    support_mask,
                    mask,
                ),
            )

            proposal = cv2.bitwise_or(
                proposal,
                mask,
            )

        after = int(
            cv2.countNonZero(
                proposal
            )
        )

        if after <= before:
            break

        mask = proposal
        iterations_used += 1

        if after >= target_pixel_area:
            break

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )

    if not contours:
        return (
            current_poly,
            final_angle,
            0.0,
            iterations_used,
        )

    contour = max(
        contours,
        key=cv2.contourArea,
    )

    if len(contour) < 3:
        return (
            current_poly,
            final_angle,
            0.0,
            iterations_used,
        )

    epsilon = (
        POLYGON_SIMPLIFICATION_FACTOR
        * cv2.arcLength(
            contour,
            True,
        )
    )

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True,
    )

    result = []

    for p in approx:
        u = float(
            np.clip(
                p[0][0] + x0,
                0,
                patch.camera.width - 1,
            )
        )

        v = float(
            np.clip(
                p[0][1] + y0,
                0,
                patch.camera.height - 1,
            )
        )

        result.append(
            (u, v)
        )

    if len(result) < 3:
        return (
            current_poly,
            final_angle,
            0.0,
            iterations_used,
        )

    patch.current_polygon = list(
        result
    )

    support_coverage = (
        float(
            cv2.countNonZero(
                cv2.bitwise_and(
                    mask,
                    support_mask,
                )
            )
        )
        / max(
            float(
                cv2.countNonZero(
                    mask
                )
            ),
            1.0,
        )
    )

    return (
        result,
        final_angle,
        support_coverage,
        iterations_used,
    )



def mean_region_geometry(patch):
    ids = np.asarray(
        sorted(patch.selected),
        dtype=int,
    )

    normals = patch.normals[
        ids
    ]

    mean_n = np.mean(
        normals,
        axis=0,
    )

    nrm = np.linalg.norm(
        mean_n
    )

    if nrm > 1e-12:
        mean_n /= nrm

    az = real_azimuth_from_normal(
        mean_n
    )

    return (
        mean_n,
        az,
    )


# ============================================================
# PREPARAR CORROSÕES
# ============================================================

corrosion_df = MASTER_DF[
    MASTER_DF[COL_TYPE]
    .astype(str)
    .str.strip()
    .str.lower()
    == "corrosion"
].copy()

valid_rows = []

for _, row in corrosion_df.iterrows():
    area = safe_float(
        row.get(COL_AREA),
        np.nan,
    )

    poly = parse_polygon_string(
        row.get(COL_POLY)
    )

    X = parse_xyz(
        row.get(COL_XYZ)
    )

    cam = CAMERAS.get(
        image_stem_any(
            row.get(COL_IMAGE)
        )
    )

    if (
        np.isfinite(area)
        and area > 0
        and len(poly) >= 3
        and X is not None
        and cam is not None
    ):
        valid_rows.append(row)

if MAX_ANOMALIES is not None:
    valid_rows = valid_rows[
        : int(MAX_ANOMALIES)
    ]

status(
    f"[8/10] Corrosões válidas para simulação 3D: "
    f"{len(valid_rows):,}"
)


# ============================================================
# SIMULAR CADA ANOMALIA AO LONGO DOS 20 PASSOS
# ============================================================

# step_results[step_index][excel_row] = dict(...)
step_results = [
    {}
    for _ in range(
        len(ENVIRONMENT)
    )
]

failures = []

status(
    "[8/10] A iniciar simulação 3D por anomalia..."
)

simulation_t0 = time.perf_counter()

for anomaly_index, row in enumerate(
    valid_rows,
    start=1,
):
    if (
        anomaly_index == 1
        or anomaly_index % PROGRESS_EVERY == 0
        or anomaly_index == len(valid_rows)
    ):
        status(
            f"[3D] Anomalia "
            f"{anomaly_index:,}/{len(valid_rows):,} | "
            f"{row.get(COL_ID)}"
        )

    try:
        patch = build_surface_patch(
            row
        )

        for step_idx, env_row in ENVIRONMENT.iterrows():
            growth = grow_patch_one_step(
                patch,
                env_row,
            )

            (
                polygon_2d,
                final_growth_angle,
                geometry_support_coverage,
                morph_iterations,
            ) = realistic_polygon_from_3d(
                patch,
                growth,
            )

            mean_n, mean_az = mean_region_geometry(
                patch
            )

            result = {
                COL_POLY: polygon_to_string(
                    polygon_2d
                ),
                COL_AREA: round(
                    growth["area_after_m2"]
                    * 10000.0,
                    AREA_DECIMALS,
                ),
                "SimulationDate": pd.Timestamp(
                    env_row["end"]
                ).strftime("%Y-%m-%d"),
                "Surface3DArea_cm2": round(
                    growth["area_after_m2"]
                    * 10000.0,
                    AREA_DECIMALS,
                ),
                "Added3DArea_cm2": round(
                    growth["added_area_m2"]
                    * 10000.0,
                    AREA_DECIMALS,
                ),
                "SurfaceVertexCount": int(
                    len(patch.selected)
                ),
                "SurfaceNormalX": float(
                    mean_n[0]
                ),
                "SurfaceNormalY": float(
                    mean_n[1]
                ),
                "SurfaceNormalZ": float(
                    mean_n[2]
                ),
                "SurfaceAzimuthDeg": (
                    float(mean_az)
                    if np.isfinite(mean_az)
                    else None
                ),
                "EnvironmentFactor": float(
                    env_row["environment_factor"]
                ),
                "WindDirectionDeg": float(
                    env_row["wind_direction_deg"]
                ),
                "WDRDirectionDeg": float(
                    env_row["wdr_direction_deg"]
                ),
                "ProximityFactor": float(
                    patch.proximity["factor"]
                ),
                "DistanceWater_m": float(
                    patch.proximity["distance_water_m"]
                ),
                "DistanceDamage_m": float(
                    patch.proximity["distance_damage_m"]
                ),
                "CorrosionNeighbors": int(
                    patch.proximity["corrosion_neighbors"]
                ),
                "RelativeGrowthTarget": float(
                    growth["relative_growth_target"]
                ),
                "TemporalGrowthFactor": float(
                    growth["temporal_growth_factor"]
                ),
                "OriginalShapeAngleDeg": float(
                    patch.shape_trend.angle_deg
                ),
                "OriginalElongationRatio": float(
                    patch.shape_trend.elongation_ratio
                ),
                "ShapeTrendStrength": float(
                    patch.shape_trend.trend_strength
                ),
                "FinalGrowthAngleDeg": float(
                    final_growth_angle
                ),
                "GeometrySupportCoverage": float(
                    geometry_support_coverage
                ),
                "MorphIterations": int(
                    morph_iterations
                ),
                "MeanFrontierExposure": float(
                    growth["mean_frontier_exposure"]
                ),
                "AddedVertices3D": int(
                    growth["added_vertices"]
                ),
                "MeanTransitionUsedDeg": float(
                    growth["mean_transition_used_deg"]
                ),
                "MaxTransitionUsedDeg": float(
                    growth["max_transition_used_deg"]
                ),
                "SurfacePatchVertices": int(
                    len(patch.points)
                ),
                "SurfaceSpacing_m": float(
                    patch.spacing_m
                ),
                "SurfaceGrowthMethod": (
                    "Hybrid: 3D surface graph + realistic 2D morphology"
                ),
            }

            step_results[
                step_idx
            ][
                int(row["_excel_row"])
            ] = result

    except Exception as exc:
        failures.append({
            "row": int(row["_excel_row"]),
            "id": str(row.get(COL_ID)),
            "error": f"{type(exc).__name__}: {exc}",
        })

        if len(failures) <= 20:
            print(
                f"[AVISO] {row.get(COL_ID)}: "
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )

status(
    f"[8/10] Simulação 3D concluída em "
    f"{fmt_time(time.perf_counter() - simulation_t0)} | "
    f"falhas={len(failures):,}"
)


# ============================================================
# ESCREVER EXCELS TEMPORAIS
# ============================================================

OUTPUT_COLUMNS = [
    "SimulationDate",
    "Surface3DArea_cm2",
    "Added3DArea_cm2",
    "SurfaceVertexCount",
    "SurfaceNormalX",
    "SurfaceNormalY",
    "SurfaceNormalZ",
    "SurfaceAzimuthDeg",
    "EnvironmentFactor",
    "WindDirectionDeg",
    "WDRDirectionDeg",
    "ProximityFactor",
    "DistanceWater_m",
    "DistanceDamage_m",
    "CorrosionNeighbors",
    "RelativeGrowthTarget",
    "TemporalGrowthFactor",
    "OriginalShapeAngleDeg",
    "OriginalElongationRatio",
    "ShapeTrendStrength",
    "FinalGrowthAngleDeg",
    "GeometrySupportCoverage",
    "MorphIterations",
    "MeanFrontierExposure",
    "AddedVertices3D",
    "MeanTransitionUsedDeg",
    "MaxTransitionUsedDeg",
    "SurfacePatchVertices",
    "SurfaceSpacing_m",
    "SurfaceGrowthMethod",
]


def header_map(ws):
    return {
        str(cell.value).strip(): cell.column
        for cell in ws[1]
        if cell.value is not None
    }


def ensure_columns(ws, names):
    headers = header_map(ws)

    next_col = ws.max_column + 1

    for name in names:
        if name not in headers:
            ws.cell(
                row=1,
                column=next_col,
                value=name,
            )
            headers[name] = next_col
            next_col += 1

    return headers


def set_by_header(ws, row, headers, name, value):
    if name not in headers:
        raise KeyError(
            f"Coluna '{name}' não encontrada."
        )

    ws.cell(
        row=row,
        column=headers[name],
        value=value,
    )


def get_by_header(ws, row, headers, name):
    if name not in headers:
        return None

    return ws.cell(
        row=row,
        column=headers[name],
    ).value


def mirror_key(ws, row, headers):
    return (
        str(
            get_by_header(
                ws,
                row,
                headers,
                COL_ID,
            )
        ),
        image_stem_any(
            get_by_header(
                ws,
                row,
                headers,
                COL_IMAGE,
            )
        ),
    )


def sync_corrosion_sheet(wb):
    if (
        MASTER_SHEET_NAME not in wb.sheetnames
        or MIRROR_CORROSION_SHEET not in wb.sheetnames
    ):
        return 0

    src = wb[
        MASTER_SHEET_NAME
    ]

    dst = wb[
        MIRROR_CORROSION_SHEET
    ]

    all_columns = [
        COL_POLY,
        COL_AREA,
    ] + OUTPUT_COLUMNS

    sh = ensure_columns(
        src,
        all_columns,
    )

    dh = ensure_columns(
        dst,
        all_columns,
    )

    index = {}

    for r in range(
        2,
        src.max_row + 1,
    ):
        if norm_type(
            get_by_header(
                src,
                r,
                sh,
                COL_TYPE,
            )
        ) != "corrosion":
            continue

        index[
            mirror_key(
                src,
                r,
                sh,
            )
        ] = r

    synced = 0

    for r in range(
        2,
        dst.max_row + 1,
    ):
        key = mirror_key(
            dst,
            r,
            dh,
        )

        src_r = index.get(
            key
        )

        if src_r is None:
            continue

        for col in all_columns:
            if (
                col in sh
                and col in dh
            ):
                set_by_header(
                    dst,
                    r,
                    dh,
                    col,
                    get_by_header(
                        src,
                        src_r,
                        sh,
                        col,
                    ),
                )

        synced += 1

    return synced


status("[9/10] A escrever os Excels temporais 3D...")

for step_idx, env_row in ENVIRONMENT.iterrows():
    sim_date = pd.Timestamp(
        env_row["end"]
    )

    status(
        f"[OUTPUT] Passo {step_idx + 1}/{len(ENVIRONMENT)} | "
        f"{sim_date.date()}"
    )

    wb = load_workbook(
        BASE_EXCEL_PATH
    )

    ws = wb[
        MASTER_SHEET_NAME
    ]

    headers = ensure_columns(
        ws,
        OUTPUT_COLUMNS,
    )

    results = step_results[
        step_idx
    ]

    for excel_row, result in results.items():
        set_by_header(
            ws,
            excel_row,
            headers,
            COL_POLY,
            result[COL_POLY],
        )

        set_by_header(
            ws,
            excel_row,
            headers,
            COL_AREA,
            result[COL_AREA],
        )

        for col in OUTPUT_COLUMNS:
            set_by_header(
                ws,
                excel_row,
                headers,
                col,
                result.get(col),
            )

    synced = sync_corrosion_sheet(
        wb
    )

    # --------------------------------------------------------
    # Environmental_Forcing
    # --------------------------------------------------------
    if "Environmental_Forcing" in wb.sheetnames:
        del wb[
            "Environmental_Forcing"
        ]

    ws_env = wb.create_sheet(
        "Environmental_Forcing"
    )

    env_until_now = ENVIRONMENT.iloc[
        : step_idx + 1
    ].copy()

    for r in dataframe_to_rows(
        env_until_now,
        index=False,
        header=True,
    ):
        ws_env.append(
            list(r)
        )

    out_path = (
        OUTPUT_DIR
        / (
            f"{BASE_EXCEL_PATH.stem}_"
            f"{sim_date.strftime('%Y-%m-%d')}_"
            f"ambiente_3D_HIBRIDO.xlsx"
        )
    )

    wb.save(
        out_path
    )

    status(
        f"[OUTPUT] Guardado: {out_path.name} | "
        f"anomalias 3D={len(results):,} | "
        f"mirror={synced:,}"
    )


# ============================================================
# LOG DE FALHAS
# ============================================================

if failures:
    fail_path = (
        OUTPUT_DIR
        / "surface_growth_failures.csv"
    )

    pd.DataFrame(
        failures
    ).to_csv(
        fail_path,
        index=False,
    )

    status(
        f"[10/10] Falhas guardadas em: {fail_path}"
    )
else:
    status(
        "[10/10] Nenhuma falha na construção/crescimento 3D."
    )


status(
    f"CONCLUÍDO. Outputs em: {OUTPUT_DIR}"
)


## csv com dados climáticos

In [4]:
# ============================================================
# EXPORTAR DADOS CLIMÁTICOS USADOS NO MODELO DE CORROSÃO
# ============================================================
#
# GERA DOIS CSVs:
#
# 1) weather_hourly_2016_2025.csv
#    -> dados horários brutos utilizados pelo modelo
#
# 2) climatic_forcing_6months_2016_2026.csv
#    -> dados agregados de 6 em 6 meses + EnvironmentFactor
#       exatamente com a mesma lógica do modelo 3D híbrido
#
# REQUISITOS:
#   pip install pandas numpy requests
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import requests


# ============================================================
# CONFIGURAÇÃO
# ============================================================

SITE_NAME = "R. dos Trabalhadores do Arco 20, 4780-424 Santo Tirso"

SITE_LAT = 41.33892
SITE_LON = -8.48144

BASE_STATE_DATE = pd.Timestamp("2016-01-01")
SIM_END_DATE = pd.Timestamp("2026-01-01")

STEP_MONTHS = 6

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

# Pasta onde guardar os CSVs
OUTPUT_DIR = Path(
    r"C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso"
    r"\Processing\Exports\Climatic_Data"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HOURLY_CSV = (
    OUTPUT_DIR
    / "weather_hourly_2016_2025.csv"
)

FORCING_CSV = (
    OUTPUT_DIR
    / "climatic_forcing_6months_2016_2026.csv"
)


# ============================================================
# PESOS EXATOS USADOS NO MODELO
# ============================================================

ENV_WEIGHTS = {
    "time_of_wetness_h": 0.30,
    "precipitation_mm": 0.15,
    "wdr_potential": 0.25,
    "rh_mean": 0.10,
    "wind_p95_ms": 0.10,
    "temperature_factor": 0.10,
}


# ============================================================
# AUXILIARES
# ============================================================

def robust_ratio(value, reference):
    """
    Normalização robusta usada no modelo.

    Limita cada variável entre 0.35 e 2.50 vezes
    a mediana dos 20 períodos.
    """

    ref = max(
        float(reference),
        1e-9,
    )

    return float(
        np.clip(
            float(value) / ref,
            0.35,
            2.50,
        )
    )


def circular_weighted_mean_deg(
    direction_deg,
    weights,
):
    """
    Média circular ponderada.

    Convenção:
      0°   = Norte
      90°  = Este
      180° = Sul
      270° = Oeste
    """

    d = np.asarray(
        direction_deg,
        dtype=float,
    )

    w = np.asarray(
        weights,
        dtype=float,
    )

    valid = (
        np.isfinite(d)
        & np.isfinite(w)
        & (w > 0)
    )

    if not np.any(valid):
        return np.nan

    rad = np.deg2rad(
        d[valid]
    )

    ww = w[valid]

    x = np.sum(
        ww * np.sin(rad)
    )

    y = np.sum(
        ww * np.cos(rad)
    )

    if abs(x) + abs(y) < 1e-12:
        return np.nan

    return float(
        np.rad2deg(
            np.arctan2(
                x,
                y,
            )
        ) % 360.0
    )


def temperature_corrosion_factor(temp_c):
    """
    Fator empírico de temperatura usado no modelo.

    NOTA:
    É um fator de aumento de dados, não uma lei física
    calibrada de corrosão.
    """

    t = float(temp_c)

    if t <= 0:
        return 0.55

    if t <= 20:
        return (
            0.55
            + 0.45 * (t / 20.0)
        )

    if t <= 30:
        return 1.0

    return max(
        0.70,
        1.0 - 0.03 * (t - 30.0),
    )


def build_periods():
    """
    Cria os mesmos 20 períodos de 6 meses do modelo.
    """

    starts = pd.date_range(
        BASE_STATE_DATE,
        SIM_END_DATE,
        freq=f"{STEP_MONTHS}MS",
        inclusive="left",
    )

    periods = []

    for start in starts:

        end = (
            pd.Timestamp(start)
            + pd.DateOffset(
                months=STEP_MONTHS
            )
        )

        periods.append(
            (
                pd.Timestamp(start),
                min(
                    pd.Timestamp(end),
                    SIM_END_DATE,
                ),
            )
        )

    return periods


PERIODS = build_periods()


# ============================================================
# DOWNLOAD OPEN-METEO
# ============================================================

def download_open_meteo_hourly(
    start_date,
    end_date,
):
    """
    Descarrega exatamente as variáveis usadas pelo modelo:
      - temperatura
      - humidade relativa
      - precipitação
      - velocidade do vento
      - direção do vento
      - radiação de onda curta
    """

    chunks = []

    for year in range(
        start_date.year,
        end_date.year + 1,
    ):

        s = max(
            pd.Timestamp(
                f"{year}-01-01"
            ),
            start_date,
        )

        e = min(
            pd.Timestamp(
                f"{year}-12-31 23:00"
            ),
            end_date
            - pd.Timedelta(
                hours=1
            ),
        )

        if e < s:
            continue

        params = {
            "latitude": SITE_LAT,
            "longitude": SITE_LON,

            "start_date":
                s.strftime(
                    "%Y-%m-%d"
                ),

            "end_date":
                e.strftime(
                    "%Y-%m-%d"
                ),

            "hourly": ",".join([
                "temperature_2m",
                "relative_humidity_2m",
                "precipitation",
                "wind_speed_10m",
                "wind_direction_10m",
                "shortwave_radiation",
            ]),

            "timezone":
                "Europe/Lisbon",

            "wind_speed_unit":
                "ms",
        }

        print(
            f"[DOWNLOAD] {year}: "
            f"{params['start_date']} -> "
            f"{params['end_date']}"
        )

        response = requests.get(
            OPEN_METEO_URL,
            params=params,
            timeout=120,
        )

        response.raise_for_status()

        payload = response.json()

        if "hourly" not in payload:

            raise RuntimeError(
                f"Resposta Open-Meteo "
                f"sem dados horários: "
                f"{payload}"
            )

        frame = pd.DataFrame(
            payload["hourly"]
        )

        if frame.empty:

            raise RuntimeError(
                f"Sem dados para {year}."
            )

        frame[
            "timestamp"
        ] = pd.to_datetime(
            frame["time"]
        )

        frame = frame.drop(
            columns=["time"]
        )

        chunks.append(
            frame
        )

    if not chunks:

        raise RuntimeError(
            "Não foram obtidos "
            "dados meteorológicos."
        )

    hourly = (
        pd.concat(
            chunks,
            ignore_index=True,
        )
        .sort_values(
            "timestamp"
        )
        .drop_duplicates(
            "timestamp"
        )
        .reset_index(
            drop=True
        )
    )

    return hourly


# ============================================================
# AGREGAR PARA OS 20 PERÍODOS USADOS NO MODELO
# ============================================================

def aggregate_environment(hourly):

    hourly = hourly.copy()

    hourly[
        "timestamp"
    ] = pd.to_datetime(
        hourly["timestamp"]
    )

    # --------------------------------------------------------
    # TIME OF WETNESS
    # --------------------------------------------------------
    # Exatamente como no código do modelo:
    # RH >= 80% e temperatura > 0°C
    #
    # 1 linha = 1 hora, logo a soma = número de horas.
    # --------------------------------------------------------

    hourly[
        "wet_hour"
    ] = (
        (
            hourly[
                "relative_humidity_2m"
            ] >= 80.0
        )
        & (
            hourly[
                "temperature_2m"
            ] > 0.0
        )
    ).astype(
        float
    )

    # --------------------------------------------------------
    # WIND-DRIVEN RAIN POTENTIAL
    # --------------------------------------------------------
    # Proxy usada no modelo:
    #
    # precipitação × velocidade do vento
    # --------------------------------------------------------

    hourly[
        "wdr_hour"
    ] = (
        hourly[
            "precipitation"
        ].clip(
            lower=0
        )
        *
        hourly[
            "wind_speed_10m"
        ].clip(
            lower=0
        )
    )

    rows = []

    for step, (
        start,
        end,
    ) in enumerate(
        PERIODS,
        start=1,
    ):

        period = hourly[
            (
                hourly[
                    "timestamp"
                ] >= start
            )
            &
            (
                hourly[
                    "timestamp"
                ] < end
            )
        ].copy()

        if period.empty:

            raise RuntimeError(
                f"Sem dados para "
                f"{start} -> {end}"
            )

        # Ponderação da direção da chuva pelo produto:
        # precipitação × vento
        rain_w = (
            period[
                "precipitation"
            ].clip(
                lower=0
            )
            *
            period[
                "wind_speed_10m"
            ].clip(
                lower=0
            )
        )

        # Direção média do vento ponderada pela velocidade
        wind_direction = (
            circular_weighted_mean_deg(
                period[
                    "wind_direction_10m"
                ],
                period[
                    "wind_speed_10m"
                ].clip(
                    lower=0.05
                ),
            )
        )

        # Direção da WDR ponderada pela chuva × vento
        wdr_direction = (
            circular_weighted_mean_deg(
                period[
                    "wind_direction_10m"
                ],
                rain_w.clip(
                    lower=0
                ),
            )
        )

        if not np.isfinite(
            wind_direction
        ):
            wind_direction = 0.0

        if not np.isfinite(
            wdr_direction
        ):
            wdr_direction = (
                wind_direction
            )

        temperature_mean = float(
            period[
                "temperature_2m"
            ].mean()
        )

        rows.append({

            "step":
                step,

            "start":
                start,

            "end":
                end,

            "hours_available":
                len(period),

            "temperature_mean_c":
                temperature_mean,

            "temperature_factor":
                temperature_corrosion_factor(
                    temperature_mean
                ),

            "rh_mean":
                float(
                    period[
                        "relative_humidity_2m"
                    ].mean()
                ),

            "time_of_wetness_h":
                float(
                    period[
                        "wet_hour"
                    ].sum()
                ),

            "precipitation_mm":
                float(
                    period[
                        "precipitation"
                    ].sum()
                ),

            "wind_mean_ms":
                float(
                    period[
                        "wind_speed_10m"
                    ].mean()
                ),

            "wind_p95_ms":
                float(
                    period[
                        "wind_speed_10m"
                    ].quantile(
                        0.95
                    )
                ),

            "wind_direction_deg":
                float(
                    wind_direction
                ),

            "wdr_potential":
                float(
                    period[
                        "wdr_hour"
                    ].sum()
                ),

            "wdr_direction_deg":
                float(
                    wdr_direction
                ),

            "shortwave_mean_wm2":
                float(
                    period[
                        "shortwave_radiation"
                    ].mean()
                ),
        })

    env = pd.DataFrame(
        rows
    )

    # ========================================================
    # REFERÊNCIAS = MEDIANAS DOS 20 PERÍODOS
    # ========================================================

    ref_columns = [
        "time_of_wetness_h",
        "precipitation_mm",
        "wdr_potential",
        "rh_mean",
        "wind_p95_ms",
        "temperature_factor",
    ]

    references = {
        column:
            float(
                env[
                    column
                ].median()
            )
        for column
        in ref_columns
    }

    # Guardar também no CSV as razões individuais,
    # para ser transparente de onde vem o EnvironmentFactor.
    for column in ref_columns:

        env[
            f"{column}_ratio"
        ] = env[
            column
        ].apply(
            lambda value,
            c=column:
                robust_ratio(
                    value,
                    references[c],
                )
        )

    # ========================================================
    # ENVIRONMENT FACTOR
    # ========================================================

    env[
        "environment_factor"
    ] = 0.0

    for column, weight in (
        ENV_WEIGHTS.items()
    ):

        env[
            "environment_factor"
        ] += (
            float(weight)
            * env[
                f"{column}_ratio"
            ]
        )

    return (
        env,
        references,
    )


# ============================================================
# EXECUÇÃO
# ============================================================

print("=" * 78)
print("EXPORTAÇÃO DOS DADOS CLIMÁTICOS USADOS PELO MODELO")
print("=" * 78)

print(
    f"Local:       {SITE_NAME}"
)

print(
    f"Latitude:    {SITE_LAT}"
)

print(
    f"Longitude:   {SITE_LON}"
)

print(
    f"Período:     "
    f"{BASE_STATE_DATE.date()} -> "
    f"{SIM_END_DATE.date()}"
)

print(
    f"Passo:       "
    f"{STEP_MONTHS} meses"
)

print("=" * 78)


# ------------------------------------------------------------
# 1) DADOS HORÁRIOS
# ------------------------------------------------------------

if HOURLY_CSV.exists():

    print(
        f"[INFO] Já existe cache horário:"
        f"\n       {HOURLY_CSV}"
    )

    hourly = pd.read_csv(
        HOURLY_CSV,
        parse_dates=[
            "timestamp"
        ],
    )

else:

    hourly = (
        download_open_meteo_hourly(
            BASE_STATE_DATE,
            SIM_END_DATE,
        )
    )

    hourly.to_csv(
        HOURLY_CSV,
        index=False,
    )

    print(
        f"\n[OK] CSV horário guardado:"
        f"\n     {HOURLY_CSV}"
    )


# ------------------------------------------------------------
# 2) AGREGAÇÃO EXATAMENTE COMO NO MODELO
# ------------------------------------------------------------

forcing, references = (
    aggregate_environment(
        hourly
    )
)


# ------------------------------------------------------------
# Acrescentar metadata útil a cada linha
# ------------------------------------------------------------

forcing.insert(
    0,
    "site_name",
    SITE_NAME,
)

forcing.insert(
    1,
    "latitude",
    SITE_LAT,
)

forcing.insert(
    2,
    "longitude",
    SITE_LON,
)


# ------------------------------------------------------------
# GUARDAR FORCING SEMESTRAL
# ------------------------------------------------------------

forcing.to_csv(
    FORCING_CSV,
    index=False,
)

print(
    f"\n[OK] CSV climático semestral guardado:"
    f"\n     {FORCING_CSV}"
)


# ============================================================
# RESUMO
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "RESUMO"
)

print(
    "=" * 78
)

print(
    f"Horas meteorológicas: "
    f"{len(hourly):,}"
)

print(
    f"Períodos semestrais:  "
    f"{len(forcing)}"
)

print(
    "\nMedianas usadas para normalização:"
)

for key, value in (
    references.items()
):

    print(
        f"  {key:25s} = "
        f"{value:.6f}"
    )

print(
    "\nPesos do EnvironmentFactor:"
)

for key, value in (
    ENV_WEIGHTS.items()
):

    print(
        f"  {key:25s} = "
        f"{value:.2f}"
    )

print(
    "\nEnvironmentFactor por período:"
)

for _, row in forcing.iterrows():

    print(
        f"  Step "
        f"{int(row['step']):02d} | "
        f"{pd.Timestamp(row['start']).date()} -> "
        f"{pd.Timestamp(row['end']).date()} | "
        f"Env={row['environment_factor']:.4f}"
    )

print(
    "\nCONCLUÍDO."
)


EXPORTAÇÃO DOS DADOS CLIMÁTICOS USADOS PELO MODELO
Local:       R. dos Trabalhadores do Arco 20, 4780-424 Santo Tirso
Latitude:    41.33892
Longitude:   -8.48144
Período:     2016-01-01 -> 2026-01-01
Passo:       6 meses
[DOWNLOAD] 2016: 2016-01-01 -> 2016-12-31
[DOWNLOAD] 2017: 2017-01-01 -> 2017-12-31
[DOWNLOAD] 2018: 2018-01-01 -> 2018-12-31
[DOWNLOAD] 2019: 2019-01-01 -> 2019-12-31
[DOWNLOAD] 2020: 2020-01-01 -> 2020-12-31
[DOWNLOAD] 2021: 2021-01-01 -> 2021-12-31
[DOWNLOAD] 2022: 2022-01-01 -> 2022-12-31
[DOWNLOAD] 2023: 2023-01-01 -> 2023-12-31
[DOWNLOAD] 2024: 2024-01-01 -> 2024-12-31
[DOWNLOAD] 2025: 2025-01-01 -> 2025-12-31

[OK] CSV horário guardado:
     C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\Processing\Exports\Climatic_Data\weather_hourly_2016_2025.csv

[OK] CSV climático semestral guardado:
     C:\Users\bolseiro\Documents\DIGIMATRIA\Obras\Santo Tirso\Processing\Exports\Climatic_Data\climatic_forcing_6months_2016_2026.csv

RESUMO
Horas meteorológicas: 87,